### **Student Information**
GitHub ID: TyrannosaurusR

Kaggle name: TRex9527

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

---

## 1. Model Development (10 pts Required)

### 1.1 Preprocessing Steps

The four data files were merged on `post_id` and split by the `set` column into 3,360 training posts and 1,440 test posts. The header of `sampleSubmission.csv` reads `post_id`, whereas the competition requires `id,label`; this mismatch caused the first submission to be rejected.

Three cleaning procedures were applied, one for each model line. The stronger the model, the lighter the cleaning, because a pretrained model has already been exposed to raw text.

| Line | Cleaning | Reason |
|---|---|---|
| A, TF-IDF | `[NAME]`, `@<username>` and `##URL##` are replaced by dedicated tokens; lowercase; letter runs shortened to two (soooo → soo); `#` removed | the default tokenizer splits the placeholders into ordinary words such as name and url |
| B, Transformer | only `@<username>` → `@user` and `##URL##` → `http`; case, emoji and punctuation retained | consistent with the tweet text seen by twitter-roberta during pretraining |
| C, LLM | none; the raw text is placed in the prompt | an LLM reads raw text best |

The first version of line A contained two regular-expression errors, which the AI assistant identified when it executed the code (log row 23). The pattern `r"[NAME]"` is a character set, so it replaced every capital N, A, M and E and altered 1,231 rows. The substitution `(.)\1+` → `\1` converted "good" into "god" and "1000" into "10". Both errors were corrected: placeholders are replaced as fixed strings first, and letter runs are shortened to two letters using `[a-z]`, which leaves digits unchanged.

The training set contains 50 groups of identical text, and the labels within these groups agree only 68% of the time. These rows were neither removed nor relabeled, since the correct label cannot be determined. Instead, all copies of the same text are assigned to the same fold, so that no model is validated on a text it was trained on. This label noise sets the performance ceiling of the task and is discussed further in Section 1.3.

### 1.2 Feature Engineering Steps

Line A uses TF-IDF over word 1-2 grams and char_wb 2-5 grams, combined with `FeatureUnion` inside a Pipeline. The Pipeline is refit in every fold, so the vocabulary and the IDF values never include the validation fold. TF-IDF + LR reaches an OOF macro F1 of 0.5086, and frozen bge-base sentence embeddings with LR reach 0.5559. The Transformers and LLMs use their own tokenizers, with `max_len` set to the 99th percentile of the training lengths (at most 128), and no hand-crafted features are added.

Two further attempts were not adopted. Eighteen hand-crafted style features (length, `!`, emoji, swear words, negation and others) improved the score by only +0.005, and this gain was in-sample. In addition, `post_id` is unrelated to the label (chi-square p = 0.29 to 0.66). Feature engineering therefore offers limited room for improvement, and most of the gain comes from stronger and more diverse models.

### 1.3 Explanation of Your Model

![Pipeline](./pics/pipeline.png)

All models share one `StratifiedGroupKFold` split with five folds (`folds.csv`). Each member produces out-of-fold probabilities (3,360 x 6) and test probabilities. Every ensemble choice is evaluated with nested CV, in which four folds select the parameters and the fifth fold scores them. Before the experiments, an adoption rule was fixed: the nested mean must rise by at least 0.005, and at least four of the five folds must improve.

| Member | OOF macro F1 |
|---|---|
| TF-IDF + LR | 0.5086 |
| bge-base + LR | 0.5559 |
| twitter-roberta-large + TAPT (5 seeds) | 0.6321 |
| twitter-roberta-large + self-distillation (3 seeds) | 0.6307 |
| twitter-roberta-large + pseudo-labels (8 runs) | 0.6310 |
| twitter-roberta-large + label smoothing (5 seeds) | 0.6293 |
| AfroXLMR-large (3 seeds) | 0.6108 |
| LLM committee (17 runs of untrained open LLMs) | 0.5827 |
| PPMd class-corpus compression | 0.4047 |

PPMd is a lossless text compressor. For each emotion class, the training posts of that class are joined into one corpus and cut into equal chunks of about 16 KB; every class gets the same amount of text, so that no class is favored for having more data. A new post is appended to each chunk and compressed, and the number of extra bytes needed, averaged over the chunks of a class, is the cost of assigning the post to that class. The class probability is proportional to the class prior times e^(-0.5 x cost), so a post that compresses well next to a class corpus looks like that class.

The ensemble is a fixed formula with three weights:

(i) R6 = 0.7 x mean(5 Transformers) + 0.15 x TF-IDF + 0.15 x bge;
(ii) A = (1 - w) x R6 + w x LLM committee, with temperature T;
(iii) B = (1 - v) x A + v x PPMd.

All five folds selected T = 1, w = 0.25 and v = 0.15. B reaches 0.6490 in nested CV, and A reaches 0.6447.

Two kinds of noise limit what can be measured. Identical texts agree on only 68% of their labels, which caps the attainable score. Separately, the nested-CV error on 3,360 posts is estimated at about ±0.007 to 0.01, and replacing the seed of a single member alone moved the ensemble score between 0.641 and 0.648. The adoption threshold of +0.005 was set from this seed range; it is a practical choice, not a statistical test, and a gain of that size is hard to tell apart from noise. Under this level of noise, models with fewer parameters are more stable. Nine learned combination methods were tested, including LR stacking (-0.015), a Super Learner (-0.002) and trees (-0.001), and all of them underperformed the fixed formula. Diversity, rather than individual accuracy, contributed the most. The LLM committee scores only 0.58 alone, yet it adds the largest gain (+0.009), because its errors differ from those of the Transformers. After fine-tuning, the LLMs behaved like the Transformers and lost this value. A diverse member must also be sufficiently accurate: a compression-distance member was too weak to help (OOF macro F1 0.28, gain +0.0009), whereas PPMd (0.40) was both different and accurate enough. The compression-distance member used the normalized compression distance, NCD(x, y) = (C(xy) - min(C(x), C(y))) / max(C(x), C(y)), where C is the zlib-compressed length, and classified each post by a weighted vote of its 10 nearest training posts.

| Submission | nested CV | Public | Private |
|---|---|---|---|
| R6 (no LLM) | 0.6363 | 0.6434 | 0.6581 |
| R1 (9 Transformers) | 0.6336 | 0.6330 | 0.6533 |
| A (+ LLM committee) | 0.6447 | 0.6440 | 0.6492 |
| B (+ PPMd, final) | 0.6490 | 0.6440 | 0.6501 |

The final Private score is 0.6501, ranking 24th among 103 teams. Two problems are noted. First, CV indicated that the LLM committee adds +0.009, but on the Private set every file containing the committee scored below R6, which does not contain it. Second, in the final selection R6, the only submission with a different structure, was dropped, and A and B, which differ in only 30 rows, were retained; the two selections therefore provided no hedge against each other. The error analysis shows that the weakest class is disgust (F1 0.46), and the largest confusion is disgust predicted as anger (130 times).

After the competition, it was found that about half of the posts appear in GoEmotions (train 1,660 of 3,360, test 741 of 1,440). Adding ready-made Hugging Face emotion models to B did not improve the score (-0.007 to 0), and directly looking up the GoEmotions labels would yield at most +0.043. These were post-competition experiments and were not used in any submission.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

This study used Claude Code, including subagents for literature search. The tool also operated the Kaggle CLI and controlled Chrome to run Kaggle and Colab. Code marked `# [AI draft]` was drafted by the AI and executed only after review by the author. The unmarked code (preprocessing and the TF-IDF baseline) was written by the author.

The initial plan was to write all code by hand (log row 18). After the README and the Lab 1 rules were checked, the arrangement was changed to "the AI drafts, the author reviews and executes, and the use is disclosed in Section 1.4" (row 33).

| Prompt (summary) | AI suggestion | Decision and reason |
|---|---|---|
| "Why can't anyone get past 0.68?" | identical texts agree on only 68% of labels | accepted; ±0.005 was treated as noise thereafter |
| outside idea: a binary classifier as a second judge | nested CV decreased | rejected |
| "Soft-label distillation should not hurt, right?" | the AI first judged the cost too high | the author overruled the AI estimate and ran it: +0.008 per seed |
| "Try a breakthrough method" | an untrained LLM committee | accepted, +0.009 |
| "Use a friend's Kaggle key" | not advised (one account per person) | rejected |
| "Try a definition prompt with wider definitions" | wider definitions degraded both models | tuning stopped, as agreed beforehand |
| stacking, trees, Super Learner | all negative in nested CV | rejected; more parameters performed worse |
| "Copy the train labels for identical test texts" | proxy CV -0.0028 | rejected |
| PPMd compression member | +0.0042, below the threshold | included in B as the author's own bet, with A kept as the safe selection |
| "Select B and A as final" | none | incorrect in hindsight; R6 had the best Private score |

Only models under the Apache-2.0 or MIT license were used, no external LLM API was called, and no public emotion dataset was downloaded during the competition. The pseudo-labels use the test text but not the test answers. The TA was asked about this practice and did not reply, so it was treated as permitted.

The complete records are kept outside the repository in `PROMPT_LOG_Kaggle.md` (each prompt, the main points of the reply and the decision) and `KAGGLE_對話紀錄.md` (the full conversation).

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 0. AI Prompts
Cells whose first comment line is `# [AI draft]` were drafted by an AI assistant (Claude, in Claude Code) from my
specification; I reviewed, ran and decided on each of them. Unmarked cells are my own.
The prompts, the assistant's answers and my decisions are summarised in report section 1.4; the full logs
(`PROMPT_LOG_Kaggle.md`, `KAGGLE_對話紀錄.md`) are kept next to this repository.

## 1. Preprocessing Steps

### 1.0 Load and inspect the raw data

The competition ships four files, all keyed by `post_id`:

| File | Columns | Rows | Role |
|---|---|---|---|
| `final_posts.json` | `post_id`, `text` | 4800 | text of every post (train + test) |
| `emotion.csv` | `post_id`, `label` | 3360 | gold emotion, train posts only |
| `data_identification.csv` | `post_id`, `set` | 4800 | `train` or `test` membership |
| `sampleSubmission.csv` | `post_id`, `label` | 1440 | submission format |

**Input:** the post `text`. **Output:** one of six emotions (`anger`, `disgust`, `fear`, `joy`, `sadness`, `surprise`). The submission is a CSV with header `post_id,label` and one row per test post.

In [10]:
# 1.0a Load the four raw data
import pandas as pd

data_path = "D:\課業\大三上\資料探勘 Lab2\DM2026-Lab2-Exercise\data\kaggle"

posts = pd.read_json(f"{data_path}/final_posts.json")  # post_id, text (train + test)
labels = pd.read_csv(f"{data_path}/emotion.csv")                 # post_id, label (train only)
split = pd.read_csv(f"{data_path}/data_identification.csv")      # post_id, set (train / test)
sample_sub = pd.read_csv(f"{data_path}/sampleSubmission.csv")    # post_id, label (submission format)

for name, frame in [("posts", posts), ("labels", labels), ("split", split), ("sample_sub", sample_sub)]:
    print(f"{name:<11} shape={frame.shape}  columns={list(frame.columns)}")

posts       shape=(4800, 2)  columns=['post_id', 'text']
labels      shape=(3360, 2)  columns=['post_id', 'label']
split       shape=(4800, 2)  columns=['post_id', 'set']
sample_sub  shape=(1440, 2)  columns=['post_id', 'label']


In [14]:
# 1.0b Join data and spilt them
df = posts.merge(split, on="post_id", how="inner", validate="one_to_one")
df = df.merge(labels, on="post_id", how="left", validate="one_to_one")

train_df = df[df["set"] == "train"].drop(columns="set").reset_index(drop=True)
test_df = df[df["set"] == "test"].drop(columns=["set", "label"]).reset_index(drop=True)

print("train_df:", train_df.shape, "| test_df:", test_df.shape)
print(train_df["label"].isna().sum())  # Check Label has no Nan
print(set(test_df["post_id"]) == set(sample_sub["post_id"])) # Check post id if right

train_df: (3360, 3) | test_df: (1440, 2)
0
True


In [15]:
# 1.0c What the data looks like
display(train_df.head(10))
display(test_df.head(5))

label_counts = train_df["label"].value_counts()
display(pd.DataFrame({
    "count": label_counts,
    "percent": (label_counts / len(train_df) * 100).round(1),
}))

,post_id,text,label
0,f8bv5zh,Thought it looked weird as well.,disgust
1,q0rr1l9,Bear died in a pit,disgust
2,p94rwgk,I have endured worse.,sadness
3,9u6wtvm,The pottery around Brasov is in danger,fear
4,h8ea329,"Meanwhile, our soldiers were approaching the area, and the Germans, of course, knew about it and began shooting at those who remained in the conce...",fear
5,expfslt,"Takes a lot less effort to just say “fuck off, buy your own.”",disgust
6,812hnc0,What’s worse is that the judge and businessman are old college buds.,disgust
7,icb5ntj,Some have now returned to normal life and are joining forces as national activists in a united community called the Ndbaga to continue to contribu...,joy
8,7obbc83,That’s very unfortunate for the bosses. Capitalism sure does commit genocide. Ireland. India. Indonesia. Vietnam. Iraq. Should I name more?,sadness
9,1wkiset,It's sad to see people in the fashion world neglecting their physical and mental health.,sadness


,post_id,text
0,xyzx9jp,Damn. That’s profound. Did you graduate from Harvard?
1,l3vzzxt,When I said Lagos people were broke and noisy una wan slap me through whatsapp wetin concern rich person and okada humble yaself and trek
2,g680iug,Drunk and crying in bed with my pets. :|
3,m4t8bp6,An afternoon of Big Cat vs Vol Twitter will be fun
4,ai3al6e,I still can't believe how quickly a little water can regenerate a tree.


,count,percent
label,,
joy,875,26.0
anger,674,20.1
sadness,628,18.7
disgust,454,13.5
surprise,441,13.1
fear,288,8.6


In [16]:
# 1.0d Three random examples
for label, group in train_df.groupby("label"):
    print(f"[{label}]")
    for text in group.sample(3, random_state=42)["text"]:
        print("   -", text[:150])
    print()

[anger]
   - I want to SO bad
   - @<username> The Devil will be their target
   - There are only people sent to destroy you that day says it is buy 45k earphones hahaha satans fail

[disgust]
   - It is disgusting to see a child's ball cared for.
   - I never think someone is sincere when they say why “would I care if I’m being lied to” constantly?
   - He’s creepy

[fear]
   - [NAME] is on fire!!!
   - There are wifi cuts, and suddenly you're the voice of confidence outside of happiness
   - When I see a man with a tape, I get scared.

[joy]
   - edou is a 'special woman' among the people you have introduced to date.
   - hahaha, brush then mang how much love you have and don't lose it, heheh even if you're going to move the content, he,
   - INFJ hands down "Being organised brings joy to me"

[sadness]
   - The questions that arose at the end of the film remained unanswered, which left me very disappointed.
   - I was also bullied at work. I was on lithium for a while. You're not al

In [22]:
# 1.1a clean function
import re

def clean_for_tfidf(text):
    text = text.replace("[NAME]", " nametoken ")      # nametoken
    text = text.replace("@<username>", " usertoken ") # usertoken
    text = text.replace("##URL##", " urltoken ")      # urltoken
    
    text = text.lower()
    
    text = text.replace("#", " ") # Delete # and turn into blank
    text = re.sub(r"([a-z])\1{2,}", r"\1\1", text)           # Shorten
    
    return re.sub(r"\s+", " ", text).strip()

train_df["text_tfidf"] = train_df["text"].map(clean_for_tfidf)
test_df["text_tfidf"] = test_df["text"].map(clean_for_tfidf)

print(f"train shape:{train_df.shape}")
print(f"test shape:{test_df.shape}")

display(train_df[["text", "text_tfidf"]].head())

train shape:(3360, 4)
test shape:(1440, 3)


,text,text_tfidf
0,Thought it looked weird as well.,thought it looked weird as well.
1,Bear died in a pit,bear died in a pit
2,I have endured worse.,i have endured worse.
3,The pottery around Brasov is in danger,the pottery around brasov is in danger
4,"Meanwhile, our soldiers were approaching the area, and the Germans, of course, knew about it and began shooting at those who remained in the conce...","meanwhile, our soldiers were approaching the area, and the germans, of course, knew about it and began shooting at those who remained in the conce..."


### 1.2 Transformer-line cleaning
Pretrained transformers learned from raw text, so this line changes as little as possible:
only the two placeholders whose pretraining form we know are mapped (twitter-roberta saw `@user` and `http`).
Case, emoji, punctuation, elongations and `[NAME]` are kept, because they carry emotion signal the model can read.

In [ ]:
# [AI draft]
# 1.2a Minimal cleaning for transformer / sentence-embedding models
def clean_for_transformer(text):
    text = text.replace("@<username>", "@user")   # twitter-roberta pretraining convention
    text = text.replace("##URL##", "http")        # twitter-roberta pretraining convention
    return re.sub(r"\s+", " ", text).strip()

train_df["text_tr"] = train_df["text"].map(clean_for_transformer)
test_df["text_tr"] = test_df["text"].map(clean_for_transformer)

print(f"train shape:{train_df.shape} | test shape:{test_df.shape}")
for col in ["text", "text_tr"]:
    print(f"{col:<8} '@<username>': {train_df[col].str.contains('@<username>', regex=False).sum():>4}"
          f" | '##URL##': {train_df[col].str.contains('##URL##', regex=False).sum():>3}"
          f" | '[NAME]': {train_df[col].str.contains('[NAME]', regex=False).sum():>4}")

# Show rows where the cleaning actually changed something
changed = train_df[train_df["text"] != train_df["text_tr"]]
print("changed rows:", len(changed))
display(changed[["text", "text_tr"]].head())

### 1.3 LLM-line input and label noise
The untrained LLMs (section 3.6) read each post inside a chat prompt and are scored on the next token after
`Emotion:`. The post gets the same minimal cleaning as the transformer line. The label check shows why scores
saturate: identical texts in train often carry different labels, so the folds keep identical texts together.

In [ ]:
# [AI draft]
# 1.3a The prompt every LLM in the committee sees (strings copied from llm_worker.py, section 3.6)
SYSTEM_PROMPT = "You are an expert at recognizing the emotion expressed in social media posts."
USER_PROMPT = ("Post:\n{post}\n\nWhich emotion does this post express? "
               "Answer with exactly one word: anger, disgust, fear, joy, sadness, or surprise.")
ANSWER_PREFIX = "Emotion:"   # the six label words (" anger", ...) are each one token after this prefix

example = train_df.iloc[0]
print("[system]   ", SYSTEM_PROMPT)
print("[user]     ", USER_PROMPT.format(post=example["text_tr"]).replace("\n", "\n            "))
print("[assistant]", ANSWER_PREFIX, "<next-token scores of the six label words>  | gold:", example["label"])

In [ ]:
# [AI draft]
# 1.3b Label noise: identical texts with different labels (folds keep them together, see 3.1a)
norm_text = train_df["text"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
dup = train_df.assign(norm=norm_text).groupby("norm")["label"].agg(["nunique", "size"])
dup = dup[dup["size"] > 1]
print(f"texts appearing 2+ times: {len(dup)} groups, {int(dup['size'].sum())} rows")
print(f"groups whose copies share one label: {(dup['nunique'] == 1).sum()} ({(dup['nunique'] == 1).mean():.0%})")
norm_test = test_df["text"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
print(f"test posts identical to a train post: {norm_test.isin(norm_text).sum()}")

## 2. Feature Engineering Steps

In [25]:
# 2.1a TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion

word_tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
char_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)

union = FeatureUnion([
    ("word", word_tfidf),  # for word
    ("char", char_tfidf)   # for char
])

# Union
X_train = union.fit_transform(train_df["text_tfidf"])
X_test = union.transform(test_df["text_tfidf"])

print(f"train shape:{X_train.shape}")
print(f"test shape:{X_test.shape}")

train shape:(3360, 34885)
test shape:(1440, 34885)


## 3. Model Implementation Steps

In [31]:
# 3.1a Cross-validation splitter
from sklearn.model_selection import StratifiedGroupKFold
groups = pd.factorize(train_df["text"])[0]
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=113034008)

X = train_df["text_tfidf"]
y = train_df["label"]

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups)):
    # 1. check the intersection between train and test
    train_texts = set(train_df.iloc[train_idx]["text"])
    val_texts = set(train_df.iloc[val_idx]["text"])
    overlap = len(train_texts & val_texts)
    
    # 2. check the even distribution
    val_dist = y.iloc[val_idx].value_counts(normalize=True).round(3).to_dict()
    
    print(f"Fold {fold} | train: {len(train_idx)}, test: {len(val_idx)} | overlap: {overlap}")
    print(f"   Propotion: {val_dist}")

Fold 0 | train: 2687, test: 673 | overlap: 0
   Propotion: {'joy': 0.26, 'anger': 0.201, 'sadness': 0.187, 'disgust': 0.135, 'surprise': 0.131, 'fear': 0.086}
Fold 1 | train: 2688, test: 672 | overlap: 0
   Propotion: {'joy': 0.26, 'anger': 0.201, 'sadness': 0.188, 'disgust': 0.135, 'surprise': 0.131, 'fear': 0.085}
Fold 2 | train: 2688, test: 672 | overlap: 0
   Propotion: {'joy': 0.26, 'anger': 0.199, 'sadness': 0.186, 'disgust': 0.135, 'surprise': 0.132, 'fear': 0.086}
Fold 3 | train: 2688, test: 672 | overlap: 0
   Propotion: {'joy': 0.26, 'anger': 0.201, 'sadness': 0.186, 'disgust': 0.135, 'surprise': 0.131, 'fear': 0.086}
Fold 4 | train: 2689, test: 671 | overlap: 0
   Propotion: {'joy': 0.261, 'anger': 0.201, 'sadness': 0.188, 'disgust': 0.134, 'surprise': 0.131, 'fear': 0.085}


In [45]:
# 3.1b feature define and model pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import f1_score, accuracy_score
import numpy as np
pipe = Pipeline([
    ("features", FeatureUnion([
        ("word", word_tfidf),
        ("char", char_tfidf)
    ])),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000))
])

oof_probs = cross_val_predict(pipe, X, y, cv=sgkf, groups=groups, method="predict_proba")
print("prop shape:", oof_probs.shape)
pred_idx = np.argmax(oof_probs, axis=1)
classes = np.unique(y)
pred = classes[pred_idx]
f1 = f1_score(y, pred, average="macro")
acc = accuracy_score(y, pred)
print(f"OOF Macro F1: {f1:.4f}")
print(f"OOF Accuracy: {acc:.4f}")

prop shape: (3360, 6)
OOF Macro F1: 0.5086
OOF Accuracy: 0.5220


In [ ]:
# [AI draft]
# 3.1c Save fold assignment and OOF probabilities
# Keyed by post_id so later models (e.g. transformers on Kaggle Notebook)
# can reuse the exact same folds and be aligned row-by-row for the ensemble.
from pathlib import Path

pred_dir = Path("data/kaggle/preds")   # data/kaggle/ is git-ignored
pred_dir.mkdir(parents=True, exist_ok=True)

# Fold id of every train row (same splits that cross_val_predict used)
fold_id = np.empty(len(train_df), dtype=int)
for fold, (_, val_idx) in enumerate(sgkf.split(X, y, groups)):
    fold_id[val_idx] = fold
folds_df = pd.DataFrame({"post_id": train_df["post_id"], "label": y, "fold": fold_id})
folds_df.to_csv(pred_dir / "folds.csv", index=False)

# OOF probabilities: one column per class, column order = classes (alphabetical)
oof_df = pd.DataFrame(oof_probs, columns=classes)
oof_df.insert(0, "post_id", train_df["post_id"])
oof_df.to_csv(pred_dir / "oof_tfidf_lr.csv", index=False)

# Check: reload the file and recompute the score
check = pd.read_csv(pred_dir / "oof_tfidf_lr.csv")
check_pred = classes[check[classes].to_numpy().argmax(axis=1)]
print("Rows per fold:", folds_df["fold"].value_counts().sort_index().tolist())
print("OOF file shape:", check.shape)
print(f"Reloaded OOF Macro F1: {f1_score(y, check_pred, average='macro'):.4f}")

### 3.3 Baseline submission (TF-IDF + LR)
Refit on all train rows and submit once, to secure the 0.5 baseline before the transformer runs.

In [ ]:
# [AI draft]
# 3.3 Baseline submission: refit on all 3360 train rows, predict test
pipe.fit(X, y)
test_probs = pipe.predict_proba(test_df["text_tfidf"])

# Save test probabilities for the ensemble later
test_prob_df = pd.DataFrame(test_probs, columns=pipe.classes_)
test_prob_df.insert(0, "post_id", test_df["post_id"])
test_prob_df.to_csv(pred_dir / "test_tfidf_lr.csv", index=False)

# Build submission in the same row order as sampleSubmission
submission = pd.DataFrame({
    "post_id": test_df["post_id"],
    "label": pipe.classes_[test_probs.argmax(axis=1)],
})
submission = sample_sub[["post_id"]].merge(submission, on="post_id", how="left", validate="one_to_one")

# Checks before upload
assert submission.shape == (1440, 2)
assert list(submission.columns) == ["post_id", "label"]
assert submission["label"].notna().all()
assert set(submission["label"]) <= set(classes)

# Sanity check: predicted class mix should look like the train mix
display(pd.DataFrame({
    "train %": (y.value_counts(normalize=True) * 100).round(1),
    "test pred %": (submission["label"].value_counts(normalize=True) * 100).round(1),
}))

submission.rename(columns={"post_id": "id"}).to_csv(pred_dir / "submission_tfidf_lr.csv", index=False)  # Kaggle wants id,label

### 3.4 Sentence-embedding member (frozen bge-base + LR)
A cheap second model family: frozen sentence embeddings of `text_tr`, logistic regression on the shared folds. Runs on CPU.

In [ ]:
# [AI draft]
# 3.4 Sentence-embedding + LR: config
import time
from sentence_transformers import SentenceTransformer

# ~440 MB download on first run; fallback: "sentence-transformers/all-MiniLM-L6-v2" (~90 MB)
EMB_MODEL = "BAAI/bge-base-en-v1.5"
EMB_TAG = "bge_lr"  # used in output / cache file names
EMB_C_GRID = [0.1, 0.3, 1, 3, 10]

In [ ]:
# [AI draft]
# Encode train + test with a frozen encoder; cache to .npy (CPU encoding takes minutes)
# BGE v1.5: the query instruction is only for retrieval queries; plain text for classification.
emb_train_path = pred_dir / f"emb_{EMB_TAG}_train.npy"
emb_test_path = pred_dir / f"emb_{EMB_TAG}_test.npy"

def load_cached_emb(path, n_rows):
    if not path.exists():
        return None
    arr = np.load(path)
    if arr.ndim != 2 or arr.shape[0] != n_rows:
        print(f"cache shape mismatch {path.name}: {arr.shape}, re-encoding")
        return None
    return arr

X_emb_train = load_cached_emb(emb_train_path, len(train_df))
X_emb_test = load_cached_emb(emb_test_path, len(test_df))

if X_emb_train is None or X_emb_test is None or X_emb_train.shape[1] != X_emb_test.shape[1]:
    encoder = SentenceTransformer(EMB_MODEL, device="cpu")
    t0 = time.time()
    enc_kwargs = dict(batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    X_emb_train = encoder.encode(train_df["text_tr"].tolist(), **enc_kwargs)
    X_emb_test = encoder.encode(test_df["text_tr"].tolist(), **enc_kwargs)
    print(f"encoded in {time.time() - t0:.1f}s with {EMB_MODEL}")
    np.save(emb_train_path, X_emb_train)
    np.save(emb_test_path, X_emb_test)
    del encoder
else:
    print(f"loaded cached embeddings from {pred_dir}")
# Note: cache is keyed by EMB_TAG only; change EMB_TAG (or delete the .npy) when switching EMB_MODEL
print("train emb:", X_emb_train.shape, "test emb:", X_emb_test.shape)

In [ ]:
# [AI draft]
# OOF probabilities on the shared predefined folds; small C grid
folds_df = pd.read_csv(pred_dir / "folds.csv")
fold_map = folds_df.set_index("post_id")["fold"]
emb_fold = train_df["post_id"].map(fold_map)
assert emb_fold.notna().all(), "some train post_ids missing from folds.csv"
emb_fold = emb_fold.astype(int).to_numpy()
assert (train_df["label"].to_numpy() == train_df["post_id"].map(folds_df.set_index("post_id")["label"]).to_numpy()).all()

y_emb = train_df["label"].to_numpy()

def emb_oof_probs(C):
    oof = np.zeros((len(train_df), len(classes)))
    for k in np.unique(emb_fold):
        tr, va = emb_fold != k, emb_fold == k
        clf = LogisticRegression(C=C, class_weight="balanced", max_iter=2000)
        clf.fit(X_emb_train[tr], y_emb[tr])
        assert list(clf.classes_) == list(classes)
        oof[va] = clf.predict_proba(X_emb_train[va])
    return oof

emb_results = []
emb_oofs = {}
for C in EMB_C_GRID:
    oof = emb_oof_probs(C)
    pred = np.asarray(classes)[oof.argmax(1)]
    emb_oofs[C] = oof
    emb_results.append({"C": C,
                        "macro_f1": f1_score(y_emb, pred, average="macro"),
                        "accuracy": accuracy_score(y_emb, pred)})

emb_table = pd.DataFrame(emb_results)
print(emb_table.to_string(index=False, float_format="%.4f"))
best_emb_C = emb_table.loc[emb_table["macro_f1"].idxmax(), "C"]
print(f"best C = {best_emb_C}, OOF macro F1 = {emb_table['macro_f1'].max():.4f}")

In [ ]:
# [AI draft]
# Save OOF (best C) + test probs (refit on all train), then reload and re-check F1
oof_emb = pd.DataFrame(emb_oofs[best_emb_C], columns=classes)
oof_emb.insert(0, "post_id", train_df["post_id"].to_numpy())

clf_emb = LogisticRegression(C=best_emb_C, class_weight="balanced", max_iter=2000)
clf_emb.fit(X_emb_train, y_emb)
assert list(clf_emb.classes_) == list(classes)
test_emb = pd.DataFrame(clf_emb.predict_proba(X_emb_test), columns=classes)
test_emb.insert(0, "post_id", test_df["post_id"].to_numpy())

oof_emb.to_csv(pred_dir / f"oof_{EMB_TAG}.csv", index=False)
test_emb.to_csv(pred_dir / f"test_{EMB_TAG}.csv", index=False)

# Reload check
oof_back = pd.read_csv(pred_dir / f"oof_{EMB_TAG}.csv")
test_back = pd.read_csv(pred_dir / f"test_{EMB_TAG}.csv")
assert oof_back.shape == (len(train_df), len(classes) + 1)
assert test_back.shape == (len(test_df), len(classes) + 1)
y_back = oof_back["post_id"].map(train_df.set_index("post_id")["label"])
f1_back = f1_score(y_back, np.asarray(classes)[oof_back[classes].to_numpy().argmax(1)], average="macro")
print(f"reloaded OOF macro F1 = {f1_back:.4f} (expected {emb_table['macro_f1'].max():.4f})")
assert abs(f1_back - emb_table["macro_f1"].max()) < 1e-9

### 3.5 Transformer members (GPU: Kaggle T4 x2, some seeds on Google Colab T4)
All five transformer members use **one training script**; only its CONFIG block changes. Running the second code
cell below only **writes** the script to `data/kaggle/jobs/train_transformer.py` (git-ignored) - it does not train
here, there is no GPU. On Kaggle the script was the kernel notebook; inputs were the competition data and my private
dataset `dm2026-lab2-folds` (`folds.csv` from 3.1c), plus `dm2026-lab2-pl-teacher` for the pseudo-label student (3.7).
Per fold: train on 4 folds, evaluate macro F1 on the 5th after every epoch, keep the best epoch's validation
probabilities (OOF) and its test probabilities; test = mean over the 5 fold models; one run per seed.

| member | model | CONFIG change vs. the script | seeds | where it ran | OOF macro F1 (seed mean) |
|---|---|---|---|---|---|
| twrob_large | cardiffnlp/twitter-roberta-large-2022-154m | none (LR 1.5e-5, batch 16, max 10 epochs, patience 2) | 42, 7, 2026, 11, 99 | Kaggle | 0.6289 |
| twrob_pl (joins twrob_large) | same | `PSEUDO_LABELS = True` | 42, 7, 2026 | Kaggle | 0.6288 (8 runs together: 0.6310) |
| twrob_tapt | same | `TAPT_EPOCHS = 20` (masked-LM on train texts first) | 42, 7, 2026, 11, 99 | Kaggle | 0.6321 |
| twrob_distill | same | `DISTILL_ALPHA = 0.3` (leak-free pair-wise teachers) | 42, 7, 2026 | Kaggle | 0.6307 |
| twrob_ls | same | `LABEL_SMOOTHING = 0.1` | 42 (Kaggle); 7, 2026, 11, 99 (Colab) | Kaggle + Colab | 0.6293 |
| afroxlmr_large | Davlan/afro-xlmr-large-76L | `LR = 1e-5` | 42, 7, 2026 | Kaggle | 0.6108 |

Tried but outside the top 5: bertweet_large 0.6107, DeBERTa-v3-large 0.5875, twhin_large 0.5737.
The Colab copies are the same script with one extra setup cell (upload the data, set `LAB2_DATA_DIR`).

In [ ]:
# [AI draft]
# 3.5a Where the downloaded GPU outputs live
# research/ sits next to this repository: it holds files derived from the competition data, which the rules say not
# to redistribute, so it is not pushed.
from pathlib import Path
RESEARCH = Path("..") / "research"
assert RESEARCH.exists(), f"{RESEARCH.resolve()} not found"
JOBS = Path("data/kaggle/jobs")           # git-ignored; the GPU scripts below are written here
JOBS.mkdir(parents=True, exist_ok=True)
CLASSES = list(classes)                   # alphabetical; every probability file uses this column order
print("research:", RESEARCH.resolve(), "| jobs:", JOBS)

In [ ]:
%%writefile data/kaggle/jobs/train_transformer.py
# [AI draft] GPU script (Kaggle T4 / Colab T4): writing this cell does NOT train here.
# Default CONFIG = twrob_large; the table above lists each member's changes.
# [AI draft]
# ---- CONFIG ----
MODEL_NAME = "cardiffnlp/twitter-roberta-large-2022-154m"   # alternative: "microsoft/deberta-v3-large"
TAG = "twrob_large"          # short model name for output files (e.g. "debv3_large"); files get _s<seed>
SEEDS = [42, 7, 2026]                 # one full 5-fold run per seed
MAX_LEN = None               # None -> p99 token length of train texts, capped at 128
LR = 1.5e-5                  # roberta-large: 1.5e-5 ; deberta-v3-large: 8e-6
EPOCHS = 10                  # max epochs; early stopping on val macro F1
PATIENCE = 2                 # stop after this many epochs without val improvement
BATCH_SIZE = 16              # deberta-v3-large: 8 ...
GRAD_ACCUM = 1               # ... with GRAD_ACCUM = 2 (effective batch 16)
WARMUP_RATIO = 0.1           # linear warmup over 10% of steps, then linear decay
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0          # gradient clipping
USE_FP16 = True              # mixed precision on GPU; DeBERTa-v3 may give NaN loss in fp16 -> set False
MAX_ATTEMPTS = 3              # retry a fold with a new seed if training collapses
COLLAPSE_F1 = 0.30            # best val macro F1 below this = degenerate run (e.g. predicts one class)
TAPT_EPOCHS = 0               # >0: continue masked-LM pretraining on train texts (no labels) before fine-tuning
TAPT_LR = 5e-5
DISTILL_ALPHA = 0.0           # >0: student target = (1-a)*one-hot + a*teacher probs (leak-free pair-wise teachers)
LABEL_SMOOTHING = 0.0         # >0: cross-entropy with uniform label smoothing (hard-label path only)
PSEUDO_LABELS = False          # True: add the test posts to training with teacher soft labels (no test labels used)
SMOKE_TEST = False           # True = tiny CPU run to check the code end to end

if SMOKE_TEST:
    MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"   # small model, random classification head
    TAG = "smoke"
    MAX_LEN = 32
    EPOCHS = 1
    N_SMOKE_TRAIN, N_SMOKE_TEST = 300, 100


# [AI draft]
import gc
import glob
import os
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch.utils.data import DataLoader
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = USE_FP16 and DEVICE.type == "cuda"   # fp16 autocast only on GPU
print(f"torch {torch.__version__} | transformers {transformers.__version__} | device {DEVICE} | fp16 {USE_AMP}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# [AI draft]
ON_KAGGLE = Path("/kaggle/input").exists()

def find_input(name, local_path):
    """First match of /kaggle/input/**/<name> on Kaggle, else the local path."""
    if ON_KAGGLE:
        hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
        if hits:
            return Path(hits[0])
    if Path(local_path).exists():
        return Path(local_path)
    raise FileNotFoundError(f"{name} not found (Kaggle: add it via 'Add Input'; local: {local_path})")

LOCAL_DATA_DIR = Path(os.environ.get("LAB2_DATA_DIR", "data/kaggle"))
POSTS_PATH = find_input("final_posts.json", LOCAL_DATA_DIR / "final_posts.json")
FOLDS_PATH = find_input("folds.csv", os.environ.get("LAB2_FOLDS", LOCAL_DATA_DIR / "preds" / "folds.csv"))
DATA_DIR = POSTS_PATH.parent   # emotion.csv and data_identification.csv sit next to final_posts.json

OUT_DIR = Path("/kaggle/working/preds") if ON_KAGGLE else Path(os.environ.get("LAB2_OUT_DIR", "transformer_preds"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("posts:", POSTS_PATH, "\nfolds:", FOLDS_PATH, "\noutput:", OUT_DIR.resolve())


# [AI draft]
CLASSES = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]   # alphabetical, fixed order
label2id = {c: i for i, c in enumerate(CLASSES)}
id2label = {i: c for c, i in label2id.items()}

posts = pd.read_json(POSTS_PATH)
ident = pd.read_csv(DATA_DIR / "data_identification.csv")
emotion = pd.read_csv(DATA_DIR / "emotion.csv")
df = (posts.merge(ident, on="post_id", how="inner", validate="one_to_one")
           .merge(emotion, on="post_id", how="left", validate="one_to_one"))
train_df = df[df["set"] == "train"].reset_index(drop=True)
test_df = df[df["set"] == "test"].reset_index(drop=True)

# Attach the shared fold ids (do NOT re-split: OOF must line up with the other models)
folds = pd.read_csv(FOLDS_PATH).rename(columns={"label": "fold_label"})
train_df = train_df.merge(folds[["post_id", "fold", "fold_label"]], on="post_id", how="left", validate="one_to_one")
assert train_df["fold"].notna().all(), "some train post_ids are missing from folds.csv"
assert (train_df["label"] == train_df["fold_label"]).all(), "labels in folds.csv do not match emotion.csv"
train_df["fold"] = train_df["fold"].astype(int)
assert set(train_df["label"]) <= set(CLASSES)

if SMOKE_TEST:
    train_df = train_df.sample(n=N_SMOKE_TRAIN, random_state=0).reset_index(drop=True)
    test_df = test_df.sample(n=N_SMOKE_TEST, random_state=0).reset_index(drop=True)
else:
    assert len(train_df) == 3360 and len(test_df) == 1440, (len(train_df), len(test_df))

def clean_for_transformer(text):
    text = text.replace("@<username>", "@user")   # twitter-roberta pretraining convention
    text = text.replace("##URL##", "http")        # twitter-roberta pretraining convention
    return re.sub(r"\s+", " ", text).strip()

train_df["text_tr"] = train_df["text"].map(clean_for_transformer)
test_df["text_tr"] = test_df["text"].map(clean_for_transformer)
y = train_df["label"].map(label2id).to_numpy()
fold_ids = train_df["fold"].to_numpy()

print("train", train_df.shape, "test", test_df.shape)
print("rows per fold:", np.bincount(fold_ids).tolist())


# [AI draft]
# Optional TAPT (Gururangan et al. 2020): continue masked-LM pretraining on the TRAIN texts only (no labels used),
# then fine-tune the classifier from the adapted weights. Test texts are not used.
if TAPT_EPOCHS > 0:
    from transformers import AutoModelForMaskedLM, DataCollatorForLanguageModeling
    t_tapt = time.time()
    tapt_tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    tapt_enc = tapt_tok(train_df["text_tr"].tolist(), truncation=True, max_length=128)
    tapt_items = [{"input_ids": a, "attention_mask": b} for a, b in zip(tapt_enc["input_ids"], tapt_enc["attention_mask"])]
    tapt_loader = DataLoader(tapt_items, batch_size=16, shuffle=True,
                             collate_fn=DataCollatorForLanguageModeling(tapt_tok, mlm_probability=0.15),
                             generator=torch.Generator().manual_seed(0))
    mlm = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).float().to(DEVICE)
    tapt_opt = torch.optim.AdamW(mlm.parameters(), lr=TAPT_LR, weight_decay=0.01)
    tapt_total = len(tapt_loader) * TAPT_EPOCHS
    tapt_sched = get_linear_schedule_with_warmup(tapt_opt, int(0.06 * tapt_total), tapt_total)
    tapt_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    mlm.train()
    for ep in range(TAPT_EPOCHS):
        run_loss = 0.0
        for batch in tapt_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                loss = mlm(**batch).loss
            tapt_opt.zero_grad()
            tapt_scaler.scale(loss).backward()
            tapt_scaler.unscale_(tapt_opt)
            torch.nn.utils.clip_grad_norm_(mlm.parameters(), 1.0)
            tapt_scaler.step(tapt_opt); tapt_scaler.update(); tapt_sched.step()
            run_loss += loss.item()
        print(f"TAPT epoch {ep + 1}/{TAPT_EPOCHS}: MLM loss {run_loss / len(tapt_loader):.4f}")
    tapt_dir = Path("/tmp/tapt_model")          # not under /kaggle/working, so it is not saved as output
    mlm.save_pretrained(tapt_dir); tapt_tok.save_pretrained(tapt_dir)
    del mlm, tapt_opt, tapt_sched, tapt_scaler, tapt_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    MODEL_NAME = str(tapt_dir)                  # fine-tuning below starts from the adapted weights
    print(f"TAPT done in {(time.time() - t_tapt) / 60:.1f} min -> {MODEL_NAME}")


# [AI draft]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tok_lens = np.array([len(ids) for ids in tokenizer(train_df["text_tr"].tolist())["input_ids"]])
p99 = int(np.ceil(np.percentile(tok_lens, 99)))
if MAX_LEN is None:
    MAX_LEN = min(p99, 128)
print(f"token length: median {np.median(tok_lens):.0f}, p99 {p99}, max {tok_lens.max()} -> MAX_LEN = {MAX_LEN}")
print(f"truncated train texts: {(tok_lens > MAX_LEN).sum()}")

def encode(texts, labels=None):
    """List of per-example dicts (input_ids, attention_mask, ... [, labels])."""
    enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN)
    items = [{k: enc[k][i] for k in enc.keys()} for i in range(len(texts))]
    if labels is not None:
        for item, lab in zip(items, labels):
            item["labels"] = int(lab)
    return items

train_items = encode(train_df["text_tr"], y)
test_items = encode(test_df["text_tr"])
collator = DataCollatorWithPadding(tokenizer=tokenizer)


# [AI draft]
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)

def build_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(CLASSES), id2label=id2label, label2id=label2id)
    return model.float().to(DEVICE)   # keep fp32 master weights; autocast handles fp16

def build_optimizer(model):
    decay = [p for p in model.parameters() if p.ndim >= 2]      # weight matrices
    no_decay = [p for p in model.parameters() if p.ndim < 2]    # biases, LayerNorm
    return torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY},
                              {"params": no_decay, "weight_decay": 0.0}], lr=LR)

def train_one_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    losses = []
    optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(loader):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with autocast():
            if LABEL_SMOOTHING > 0:
                labels = batch.pop("labels")
                loss = torch.nn.functional.cross_entropy(model(**batch).logits.float(), labels, label_smoothing=LABEL_SMOOTHING)
            else:
                loss = model(**batch).loss
        if not torch.isfinite(loss):
            raise RuntimeError("Loss is NaN/inf. If USE_FP16=True (esp. DeBERTa-v3), set USE_FP16 = False and rerun.")
        scaler.scale(loss / GRAD_ACCUM).backward()
        losses.append(loss.item())
        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
    return float(np.mean(losses))

@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    out = []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
        with autocast():
            logits = model(**batch).logits
        out.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(out)

def macro_f1(y_true, probs):
    return f1_score(y_true, probs.argmax(1), average="macro", labels=list(range(len(CLASSES))), zero_division=0)


# [AI draft]
def run_cv(seed):
    n_cls = len(CLASSES)
    oof = np.zeros((len(train_df), n_cls))
    test_sum = np.zeros((len(test_df), n_cls))
    test_loader = DataLoader(test_items, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator)
    fold_log = []

    for fold in sorted(np.unique(fold_ids)):
        for attempt in range(MAX_ATTEMPTS):
            fold_seed = seed + 1000 * attempt
            t0 = time.time()
            seed_everything(fold_seed)
            tr_idx = np.where(fold_ids != fold)[0]
            va_idx = np.where(fold_ids == fold)[0]
            train_loader = DataLoader([train_items[i] for i in tr_idx], batch_size=BATCH_SIZE, shuffle=True,
                                      collate_fn=collator, generator=torch.Generator().manual_seed(fold_seed))
            val_loader = DataLoader([train_items[i] for i in va_idx], batch_size=BATCH_SIZE * 2,
                                    shuffle=False, collate_fn=collator)

            model = build_model()
            optimizer = build_optimizer(model)
            steps_per_epoch = int(np.ceil(len(train_loader) / GRAD_ACCUM))
            total_steps = steps_per_epoch * EPOCHS
            scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
            scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

            best_f1, best_epoch, bad_epochs, best_test = -1.0, 0, 0, None
            for epoch in range(1, EPOCHS + 1):
                loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler)
                val_probs = predict_proba(model, val_loader)
                val_f1 = macro_f1(y[va_idx], val_probs)
                improved = val_f1 > best_f1
                print(f"  seed {fold_seed} fold {fold} epoch {epoch}: train loss {loss:.4f} | val macro F1 {val_f1:.4f}"
                      + (" *" if improved else ""))
                if improved:
                    best_f1, best_epoch, bad_epochs = val_f1, epoch, 0
                    oof[va_idx] = val_probs
                    best_test = predict_proba(model, test_loader)
                else:
                    bad_epochs += 1
                    if bad_epochs >= PATIENCE:
                        break
            if best_f1 >= COLLAPSE_F1 or attempt == MAX_ATTEMPTS - 1:
                break
            print(f"  fold {fold}: collapsed (best val macro F1 {best_f1:.4f}) -> retry with seed {seed + 1000 * (attempt + 1)}")
            del model, optimizer, scheduler, scaler
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
        test_sum += best_test

        elapsed = time.time() - t0
        fold_log.append({"fold": fold, "best_epoch": best_epoch, "val_macro_f1": best_f1, "minutes": elapsed / 60})
        print(f"fold {fold}: best epoch {best_epoch}, val macro F1 {best_f1:.4f}, {elapsed / 60:.1f} min")

        # free GPU memory before the next fold
        del model, optimizer, scheduler, scaler, train_loader, val_loader
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    n_folds = len(fold_log)
    return oof, test_sum / n_folds, pd.DataFrame(fold_log)


# [AI draft]
# Soft-label self-distillation without leakage.
# For outer fold k, a training row in fold j gets its soft label from a teacher trained on the 3 folds outside {j, k},
# so neither the student's validation fold k nor the row's own fold j was seen by that teacher.
# One teacher per unordered pair {j, k} serves both directions -> C(5,2) = 10 teachers.
# The teacher's best epoch for fold-j predictions is chosen on fold j only (no fold-k labels involved).
def soft_ce_epoch(model, loader, optimizer, scheduler, scaler, targets):
    model.train(); losses = []
    optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(loader):
        idx = batch.pop("idx").to(DEVICE); batch.pop("labels", None)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with autocast():
            logits = model(**batch).logits
        loss = -(targets[idx] * torch.log_softmax(logits.float(), dim=-1)).sum(-1).mean()
        if not torch.isfinite(loss):
            raise RuntimeError("Loss is NaN/inf in distillation.")
        scaler.scale(loss / GRAD_ACCUM).backward(); losses.append(loss.item())
        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer); scaler.update(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
    return float(np.mean(losses))

def fit_and_track(seed, tr_idx, eval_sets, targets=None):
    """Train on tr_idx; for each named eval set keep the probs from the epoch with its own best macro F1.
    Stops when no eval set improved for PATIENCE epochs. Returns {name: (best_f1, best_epoch, probs)} and test probs
    taken at the best epoch of the eval set named 'main' (if present)."""
    seed_everything(seed)
    items = [dict(train_items[i], idx=i) for i in tr_idx]
    loader = DataLoader(items, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator, generator=torch.Generator().manual_seed(seed))
    evl = {n: DataLoader([train_items[i] for i in ix], batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator) for n, ix in eval_sets.items()}
    test_loader = DataLoader(test_items, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator)
    model = build_model(); optimizer = build_optimizer(model)
    total = int(np.ceil(len(loader) / GRAD_ACCUM)) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    if targets is None:   # hard labels as one-hot -> same loss path
        targets = torch.nn.functional.one_hot(torch.tensor(y), len(CLASSES)).float()
    targets = targets.to(DEVICE)
    best = {n: (-1.0, 0, None) for n in eval_sets}; best_test = None; bad = 0
    for epoch in range(1, EPOCHS + 1):
        loss = soft_ce_epoch(model, loader, optimizer, scheduler, scaler, targets)
        improved_any = False
        for n, ix in eval_sets.items():
            p = predict_proba(model, evl[n]); f = macro_f1(y[ix], p)
            if f > best[n][0]:
                best[n] = (f, epoch, p); improved_any = True
                if n == "main":
                    best_test = predict_proba(model, test_loader)
        print(f"    epoch {epoch}: loss {loss:.4f} | " + " ".join(f"{n} {best[n][0]:.4f}" for n in eval_sets))
        bad = 0 if improved_any else bad + 1
        if bad >= PATIENCE:
            break
    del model, optimizer, scheduler, scaler, loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return best, best_test

def run_cv_distill(seed):
    folds_all = sorted(np.unique(fold_ids)); n_cls = len(CLASSES)
    # 1) pair-wise teachers
    teach = {}   # teach[(k, j)] = teacher probs for rows of fold j, from a teacher that saw neither j nor k
    for a_i, j in enumerate(folds_all):
        for k in folds_all[a_i + 1:]:
            t0 = time.time()
            tr = np.where(~np.isin(fold_ids, [j, k]))[0]
            for attempt in range(MAX_ATTEMPTS):
                best, _ = fit_and_track(seed + 500 + 1000 * attempt, tr,
                                        {f"f{j}": np.where(fold_ids == j)[0], f"f{k}": np.where(fold_ids == k)[0]})
                if min(best[f"f{j}"][0], best[f"f{k}"][0]) >= COLLAPSE_F1 or attempt == MAX_ATTEMPTS - 1:
                    break
                print(f"  teacher {{{j},{k}}} collapsed -> retry")
            teach[(k, j)] = best[f"f{j}"][2]; teach[(j, k)] = best[f"f{k}"][2]
            print(f"teacher {{{j},{k}}}: F1 on fold {j} {best[f'f{j}'][0]:.4f}, on fold {k} {best[f'f{k}'][0]:.4f}, {(time.time()-t0)/60:.1f} min")
    # 2) students with soft targets
    onehot = np.eye(n_cls)[y]
    oof = np.zeros((len(train_df), n_cls)); test_sum = np.zeros((len(test_df), n_cls)); log = []
    for k in folds_all:
        t0 = time.time()
        T = onehot.copy()
        for j in folds_all:
            if j != k:
                rows = np.where(fold_ids == j)[0]
                T[rows] = (1 - DISTILL_ALPHA) * onehot[rows] + DISTILL_ALPHA * teach[(k, j)]
        tr, va = np.where(fold_ids != k)[0], np.where(fold_ids == k)[0]
        for attempt in range(MAX_ATTEMPTS):
            best, best_test = fit_and_track(seed + 1000 * attempt, tr, {"main": va}, torch.tensor(T, dtype=torch.float32))
            if best["main"][0] >= COLLAPSE_F1 or attempt == MAX_ATTEMPTS - 1:
                break
            print(f"  student fold {k} collapsed -> retry")
        f1, ep, p = best["main"]; oof[va] = p; test_sum += best_test
        log.append({"fold": k, "best_epoch": ep, "val_macro_f1": f1, "minutes": (time.time() - t0) / 60})
        print(f"student fold {k}: best epoch {ep}, val macro F1 {f1:.4f}, {(time.time()-t0)/60:.1f} min")
    return oof, test_sum / len(folds_all), pd.DataFrame(log)


# [AI draft]
# Pseudo-labelling with leak-free teachers.
# For fold k, the test posts are added to the training set. Their targets are the test probabilities of a teacher
# model that never saw the labels of fold k (teacher_f<k>.csv). Train rows keep their one-hot labels.
# The student is evaluated on fold k exactly as before, so its OOF score is comparable with the plain runs.
def load_teacher(fold):
    path = find_input(f"teacher_f{fold}.csv", Path(os.environ.get("LAB2_TEACHER_DIR", "teacher")) / f"teacher_f{fold}.csv")
    probs = pd.read_csv(path).drop_duplicates("post_id").set_index("post_id").loc[test_df["post_id"], CLASSES].to_numpy()
    assert probs.shape == (len(test_df), len(CLASSES)) and np.allclose(probs.sum(1), 1, atol=1e-3)
    return probs

def run_cv_pseudo(seed):
    folds_all = sorted(np.unique(fold_ids)); n_cls = len(CLASSES); n_tr = len(train_df)
    onehot = np.eye(n_cls)[y]
    no_label = lambda item: {k: v for k, v in item.items() if k != "labels"}
    pool = [no_label(it) for it in train_items] + [no_label(it) for it in test_items]   # train rows, then test rows
    test_loader = DataLoader(test_items, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator)
    oof = np.zeros((n_tr, n_cls)); test_sum = np.zeros((len(test_df), n_cls)); log = []
    for k in folds_all:
        t0 = time.time()
        targets = torch.tensor(np.vstack([onehot, load_teacher(k)]), dtype=torch.float32).to(DEVICE)
        va = np.where(fold_ids == k)[0]
        rows = np.concatenate([np.where(fold_ids != k)[0], n_tr + np.arange(len(test_df))])
        val_loader = DataLoader([train_items[i] for i in va], batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator)
        for attempt in range(MAX_ATTEMPTS):
            fold_seed = seed + 1000 * attempt
            seed_everything(fold_seed)
            loader = DataLoader([dict(pool[i], idx=i) for i in rows], batch_size=BATCH_SIZE, shuffle=True,
                                collate_fn=collator, generator=torch.Generator().manual_seed(fold_seed))
            model = build_model(); optimizer = build_optimizer(model)
            total = int(np.ceil(len(loader) / GRAD_ACCUM)) * EPOCHS
            scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total), total)
            scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
            best_f1, best_epoch, bad, best_val, best_test = -1.0, 0, 0, None, None
            for epoch in range(1, EPOCHS + 1):
                loss = soft_ce_epoch(model, loader, optimizer, scheduler, scaler, targets)
                val_probs = predict_proba(model, val_loader)
                val_f1 = macro_f1(y[va], val_probs)
                improved = val_f1 > best_f1
                print(f"  seed {fold_seed} fold {k} epoch {epoch}: train loss {loss:.4f} | val macro F1 {val_f1:.4f}" + (" *" if improved else ""))
                if improved:
                    best_f1, best_epoch, bad, best_val = val_f1, epoch, 0, val_probs
                    best_test = predict_proba(model, test_loader)
                else:
                    bad += 1
                    if bad >= PATIENCE:
                        break
            del model, optimizer, scheduler, scaler, loader
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            if best_f1 >= COLLAPSE_F1 or attempt == MAX_ATTEMPTS - 1:
                break
            print(f"  fold {k}: collapsed (best val macro F1 {best_f1:.4f}) -> retry")
        oof[va] = best_val; test_sum += best_test
        # per-fold test probabilities, so this model can itself serve as a leak-free teacher later
        fold_test = pd.DataFrame(best_test, columns=CLASSES); fold_test.insert(0, "post_id", test_df["post_id"].to_numpy())
        fold_test.to_csv(OUT_DIR / f"testfold_{TAG}_s{seed}_f{k}.csv", index=False)
        log.append({"fold": k, "best_epoch": best_epoch, "val_macro_f1": best_f1, "minutes": (time.time() - t0) / 60})
        print(f"pseudo fold {k}: best epoch {best_epoch}, val macro F1 {best_f1:.4f}, {(time.time() - t0) / 60:.1f} min")
    return oof, test_sum / len(folds_all), pd.DataFrame(log)


# [AI draft]
def to_prob_frame(post_ids, probs):
    out = pd.DataFrame(probs, columns=CLASSES)
    out.insert(0, "post_id", np.asarray(post_ids))
    return out

for seed in SEEDS:
    t_seed = time.time()
    oof, test_probs, fold_log = (run_cv_pseudo if PSEUDO_LABELS else run_cv_distill if DISTILL_ALPHA > 0 else run_cv)(seed)

    oof_pred = oof.argmax(1)
    print(f"\n=== {TAG} seed {seed} ({(time.time() - t_seed) / 60:.1f} min) ===")
    print(fold_log.to_string(index=False, float_format="%.4f"))
    print(f"OOF macro F1 {macro_f1(y, oof):.4f} | OOF accuracy {accuracy_score(y, oof_pred):.4f}")
    print(classification_report(y, oof_pred, labels=list(range(len(CLASSES))), target_names=CLASSES,
                                digits=4, zero_division=0))

    oof_path = OUT_DIR / f"oof_{TAG}_s{seed}.csv"
    test_path = OUT_DIR / f"test_{TAG}_s{seed}.csv"
    to_prob_frame(train_df["post_id"], oof).to_csv(oof_path, index=False)
    to_prob_frame(test_df["post_id"], test_probs).to_csv(test_path, index=False)

    # Reload check: shape, row sums, post_id alignment, same score
    oof_back, test_back = pd.read_csv(oof_path), pd.read_csv(test_path)
    assert list(oof_back.columns) == ["post_id"] + CLASSES
    assert oof_back.shape == (len(train_df), len(CLASSES) + 1)
    assert test_back.shape == (len(test_df), len(CLASSES) + 1)
    assert np.allclose(oof_back[CLASSES].sum(axis=1), 1, atol=1e-4)
    assert np.allclose(test_back[CLASSES].sum(axis=1), 1, atol=1e-4)
    assert (oof_back["post_id"] == train_df["post_id"]).all() and (test_back["post_id"] == test_df["post_id"]).all()
    y_back = oof_back["post_id"].map(train_df.set_index("post_id")["label"]).map(label2id).to_numpy()
    assert abs(macro_f1(y_back, oof_back[CLASSES].to_numpy()) - macro_f1(y, oof)) < 1e-6
    print("saved:", oof_path, test_path)

In [ ]:
# [AI draft]
# 3.5b Load every transformer run (OOF + test probabilities) and average the seeds of each member
import glob

def read_probs(path, ids):
    """post_id-keyed probability file -> array aligned to ids, columns in CLASSES order."""
    return pd.read_csv(path).drop_duplicates("post_id").set_index("post_id").loc[ids, CLASSES].to_numpy()

def macro_f1(y_true, pred):
    return f1_score(y_true, pred, average="macro", labels=range(len(CLASSES)))

folds_df = pd.read_csv(RESEARCH / "preds_backup" / "kg_folds" / "folds.csv")
assert (folds_df["post_id"].to_numpy() == train_df["post_id"].to_numpy()).all()    # same rows, same order as 3.1c
y_int = np.searchsorted(CLASSES, train_df["label"].to_numpy())
fold_all = folds_df["fold"].to_numpy()
train_ids, test_ids = train_df["post_id"], test_df["post_id"]

tr_files = {}
for o in sorted(glob.glob(str(RESEARCH / "preds_backup" / "kg_out" / "*" / "preds" / "oof_*.csv"))):
    if "/_" in o.replace("\\", "/").split("kg_out")[1]:
        continue                                          # an abandoned run kept for the record
    tr_files.setdefault(re.sub(r"_s\d+$", "", Path(o).stem[4:]), []).append(o)
rank = {k: macro_f1(y_int, np.mean([read_probs(p, train_ids) for p in v], 0).argmax(1)) for k, v in tr_files.items()}
TOP5 = sorted(rank, key=rank.get, reverse=True)[:5]       # ranked on the plain runs, as decided on 11/21
tr_files["twrob_large"] += sorted(glob.glob(str(RESEARCH / "llm" / "out" / "*" / "preds" / "oof_twrob_pl_s*.csv")))

OOF_TR = {k: np.mean([read_probs(p, train_ids) for p in tr_files[k]], 0) for k in TOP5}
TEST_TR = {k: np.mean([read_probs(p.replace("oof_", "test_"), test_ids) for p in tr_files[k]], 0) for k in TOP5}
display(pd.DataFrame([{"member": k, "runs": len(tr_files[k]), "OOF macro F1": round(macro_f1(y_int, OOF_TR[k].argmax(1)), 4),
                       "in top 5": k in TOP5} for k in TOP5] +
                     [{"member": k, "runs": len(v), "OOF macro F1": round(rank[k], 4), "in top 5": False}
                      for k, v in sorted(tr_files.items(), key=lambda kv: -rank[kv[0]]) if k not in TOP5]))

### 3.6 LLM committee: 17 untrained open LLMs (GPU: Kaggle T4 x2)
Each LLM reads the prompt from 1.3a and gives six class scores from its next-token distribution after `Emotion:`;
**no label is used for training** (zero-shot, or a few labelled train neighbours as examples). Only Apache-2.0 / MIT
models; no model fine-tuned on emotion data. Every run is softened with one temperature T and the 17 runs are
averaged. Its value is an independent view: alone it scores 0.58, but blended with the transformers it adds +0.009.

| kind | runs |
|---|---|
| zero-shot (11) | Qwen2.5-7B / 14B / 32B-Instruct, Qwen3-8B / 14B, Qwen3.8-27B, Mistral-Small-24B-Instruct, Phi-4, Gemma-4-31B-it, OLMo-3.1-32B-Instruct, Nanbeige4.3-33B |
| few-shot (3) | Qwen2.5-14B with 4 and 8 nearest train posts, Qwen2.5-7B with 8 (`--fewshot`) |
| broad label definitions (3) | Qwen2.5-14B, Qwen2.5-7B, Phi-4 (`--defs`) |

The same script also fine-tunes an LLM with QLoRA (Qwen2.5-7B, 5 folds, OOF 0.6279); that model was too similar to
the transformers to help the blend, so it is used only as the pseudo-label teacher (3.7). The code cell only writes
the script to `data/kaggle/jobs/llm_worker.py`; the Kaggle kernels ran it with the flags in the table.

In [ ]:
%%writefile data/kaggle/jobs/llm_worker.py
# [AI draft]
"""QLoRA fine-tuning of an open LLM as a 6-class emotion classifier.

One run of this script = one GPU working through a list of folds.

Method
  * Each post is placed in a chat prompt that asks for a one-word emotion.
  * The assistant's answer is started with "Emotion:". The class scores are the model's next-token scores for
    the six label words (" anger" / " disgust" / " fear" / " joy" / " sadness" / " surprise", one token each).
    No new classification head is added.
  * Only LoRA adapters are trained; the 4-bit quantized base model stays frozen (QLoRA, Dettmers et al. 2023).
  * Same 5 folds as every other model (folds.csv), so the out-of-fold predictions can be blended directly.
  * Per fold: after every epoch score the validation fold; whenever macro F1 improves, keep the validation
    probabilities and predict the test set at that moment (same protocol as the encoder kernels).

Outputs per fold, in --out:
  val_<tag>_s<seed>_f<fold>.csv        post_id + 6 probabilities for the rows of that validation fold
  testfold_<tag>_s<seed>_f<fold>.csv   post_id + 6 probabilities for all test rows
  log_<tag>_s<seed>_f<fold>.json       per-epoch history, timing, settings
"""
import argparse
import gc
import glob
import json
import os
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup

# ---- CONFIG (command line) ----
ap = argparse.ArgumentParser()
ap.add_argument("--model", default="Qwen/Qwen2.5-7B-Instruct", help="Hugging Face id or local folder")
ap.add_argument("--tag", default="qwen25_7b", help="short model name used in output file names")
ap.add_argument("--seed", type=int, default=42)
ap.add_argument("--folds", default="0,1,2,3,4", help="comma-separated fold ids to run in this process")
ap.add_argument("--out", default="llm_preds")
ap.add_argument("--epochs", type=int, default=3)
ap.add_argument("--lr", type=float, default=1e-4)
ap.add_argument("--batch", type=int, default=16)
ap.add_argument("--accum", type=int, default=1, help="gradient accumulation steps")
ap.add_argument("--eval_batch", type=int, default=32)
ap.add_argument("--max_post_tokens", type=int, default=128, help="posts longer than this are truncated")
ap.add_argument("--lora_r", type=int, default=16)
ap.add_argument("--lora_alpha", type=int, default=32)
ap.add_argument("--lora_dropout", type=float, default=0.1)
ap.add_argument("--warmup", type=float, default=0.1, help="share of steps used for linear warmup")
ap.add_argument("--weight_decay", type=float, default=0.01)
ap.add_argument("--no_4bit", action="store_true", help="load the base model unquantized")
ap.add_argument("--device_map_auto", action="store_true", help="spread the model over all visible GPUs (large models)")
ap.add_argument("--max_gpu_gb", type=float, default=13.0, help="with --device_map_auto: memory budget per GPU for the weights")
ap.add_argument("--no_grad_ckpt", action="store_true", help="turn gradient checkpointing off")
ap.add_argument("--smoke", type=int, default=0, help=">0: tiny run on this many train rows to check the code")
ap.add_argument("--screen", action="store_true",
                help="no fold training: zero-shot predictions for all rows, then a few training steps to measure memory/speed")
ap.add_argument("--probe_batches", default="8,16", help="batch sizes tried by --screen")
ap.add_argument("--fewshot", type=int, default=0,
                help=">0: put this many nearest labelled training posts (knn_e5.csv) in front of each question; no training on them")
ap.add_argument("--defs", action="store_true",
                help="put broad definitions of the six labels into the question (see DEFS_PROMPT)")
ap.add_argument("--no_probe", action="store_true", help="--screen without the training-step probe (zero-shot only, no LoRA)")
args = ap.parse_args()

CLASSES = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]   # alphabetical, fixed order
SYSTEM_PROMPT = "You are an expert at recognizing the emotion expressed in social media posts."
USER_PROMPT = ("Post:\n{post}\n\nWhich emotion does this post express? "
               "Answer with exactly one word: anger, disgust, fear, joy, sadness, or surprise.")
# Broad label definitions. Sources: the public GoEmotions-to-Ekman grouping (Demszky et al. 2020: which fine-grained
# emotions fall under each of the six) widened with patterns seen in the TRAINING labels (e.g. contempt and moral
# outrage are often labelled disgust). Written once, not tuned on scores.
DEFS_PROMPT = ("Post:\n{post}\n\n"
               "Which emotion does this post express? Use these broad definitions:\n"
               "- anger: anger, annoyance, irritation, frustration, rage, hostility, complaining, blaming, disapproval or "
               "criticism of someone or something, attacking sarcasm.\n"
               "- disgust: disgust, revulsion, contempt, scorn, moral outrage at someone's behaviour, finding something gross, "
               "stupid, pathetic, cringe-worthy or distasteful, dismissive eye-rolling.\n"
               "- fear: fear, nervousness, anxiety, worry, dread, panic, feeling threatened or unsafe, concern about what might happen.\n"
               "- joy: joy, happiness, amusement, laughter, excitement, enthusiasm, love, admiration, approval, agreement, "
               "gratitude, pride, relief, optimism, caring, desire.\n"
               "- sadness: sadness, grief, disappointment, regret, remorse, guilt, embarrassment, loneliness, hurt, "
               "pity or sympathy for a misfortune, feeling down or hopeless.\n"
               "- surprise: surprise, amazement, astonishment, shock, disbelief, realization, confusion, curiosity, "
               "wondering, puzzled questions about something unexpected.\n"
               "Answer with exactly one word: anger, disgust, fear, joy, sadness, or surprise.")
if args.defs:
    USER_PROMPT = DEFS_PROMPT
ANSWER_PREFIX = "Emotion:"   # the answer starts with this, so each label word (" fear") is one whole token
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
MAX_BAD_STEPS = 20   # abort if this many training steps give a NaN/inf loss

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"                      # fp16 autocast on GPU only
USE_4BIT = DEVICE.type == "cuda" and not args.no_4bit
FOLDS = [int(f) for f in args.folds.split(",")]
transformers.utils.logging.disable_progress_bar()


def log(msg):
    print(f"{time.strftime('%H:%M:%S')} {msg}", flush=True)


log(f"torch {torch.__version__} | transformers {transformers.__version__} | device {DEVICE} | 4bit {USE_4BIT} | "
    f"model {args.model} | folds {FOLDS} | seed {args.seed}")
if DEVICE.type == "cuda":
    log(f"GPU: {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")

# ---- Locate inputs (same rule as the encoder kernels) ----
ON_KAGGLE = Path("/kaggle/input").exists()


def find_input(name, local_path):
    """First match of /kaggle/input/**/<name> on Kaggle, else the local path."""
    if ON_KAGGLE:
        hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
        if hits:
            return Path(hits[0])
    if Path(local_path).exists():
        return Path(local_path)
    raise FileNotFoundError(f"{name} not found (Kaggle: add it via 'Add Input'; local: {local_path})")


LOCAL_DATA_DIR = Path(os.environ.get("LAB2_DATA_DIR", "data/kaggle"))
POSTS_PATH = find_input("final_posts.json", LOCAL_DATA_DIR / "final_posts.json")
FOLDS_PATH = find_input("folds.csv", os.environ.get("LAB2_FOLDS", LOCAL_DATA_DIR / "preds" / "folds.csv"))
DATA_DIR = POSTS_PATH.parent
OUT_DIR = Path(args.out)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Load data, attach the shared folds ----
label2id = {c: i for i, c in enumerate(CLASSES)}
posts = pd.read_json(POSTS_PATH)
ident = pd.read_csv(DATA_DIR / "data_identification.csv")
emotion = pd.read_csv(DATA_DIR / "emotion.csv")
df = (posts.merge(ident, on="post_id", how="inner", validate="one_to_one")
           .merge(emotion, on="post_id", how="left", validate="one_to_one"))
train_df = df[df["set"] == "train"].reset_index(drop=True)
test_df = df[df["set"] == "test"].reset_index(drop=True)
folds = pd.read_csv(FOLDS_PATH).rename(columns={"label": "fold_label"})
train_df = train_df.merge(folds[["post_id", "fold", "fold_label"]], on="post_id", how="left", validate="one_to_one")
assert train_df["fold"].notna().all(), "some train post_ids are missing from folds.csv"
assert (train_df["label"] == train_df["fold_label"]).all(), "labels in folds.csv do not match emotion.csv"
train_df["fold"] = train_df["fold"].astype(int)
assert len(train_df) == 3360 and len(test_df) == 1440, (len(train_df), len(test_df))

if args.smoke > 0:   # tiny subset: keeps every fold represented
    train_df = train_df.groupby("fold", group_keys=False).sample(n=args.smoke // 5, random_state=0).reset_index(drop=True)
    test_df = test_df.sample(n=max(args.smoke // 4, 8), random_state=0).reset_index(drop=True)


def clean_post(text):
    """Same minimal cleaning as the encoder line: placeholders -> plain conventions, collapse whitespace."""
    text = text.replace("@<username>", "@user").replace("##URL##", "http")
    return re.sub(r"\s+", " ", text).strip()


y = train_df["label"].map(label2id).to_numpy()
fold_ids = train_df["fold"].to_numpy()
N_TRAIN, N_TEST = len(train_df), len(test_df)
all_posts = [clean_post(t) for t in train_df["text"]] + [clean_post(t) for t in test_df["text"]]
log(f"train {N_TRAIN} rows, test {N_TEST} rows, rows per fold {np.bincount(fold_ids).tolist()}")

# ---- Tokenize: [prompt head] + [post, truncated] + [prompt tail]; the tail ends where the answer starts ----
tokenizer = AutoTokenizer.from_pretrained(args.model)
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
PLACEHOLDER = "<<<POST>>>"


def render_chat(turns):
    """Chat prompt text ending where the assistant's answer starts. enable_thinking=False selects the official
    non-thinking format of hybrid-thinking models (Qwen3.x, GLM-4.x); other templates ignore it. A template that
    rejects a system turn gets the system text at the start of the first user turn instead."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + turns
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except Exception:
        merged = [dict(turns[0], content=SYSTEM_PROMPT + "\n\n" + turns[0]["content"])] + turns[1:]
        return tokenizer.apply_chat_template(merged, tokenize=False, add_generation_prompt=True, enable_thinking=False)


rendered = render_chat([{"role": "user", "content": USER_PROMPT.format(post=PLACEHOLDER)}])
assert rendered.count(PLACEHOLDER) == 1, "chat template did not keep the post placeholder exactly once"
head_text, tail_text = rendered.split(PLACEHOLDER)
tail_text += ANSWER_PREFIX
head_ids = tokenizer(head_text, add_special_tokens=False)["input_ids"]
tail_ids = tokenizer(tail_text, add_special_tokens=False)["input_ids"]
post_ids = tokenizer(all_posts, add_special_tokens=False)["input_ids"]
n_trunc = sum(len(p) > args.max_post_tokens for p in post_ids)
heads = [head_ids] * len(post_ids)
if args.fewshot > 0:
    # Few-shot: earlier turns of the chat show the nearest labelled TRAINING posts with their answers.
    # knn_e5.csv was built so that a train post only sees posts from other folds; test posts see all train posts.
    DEMO_CHARS = 300   # long example posts are cut to keep the prompt short
    knn = pd.read_csv(find_input("knn_e5.csv", os.environ.get("LAB2_KNN", "knn/knn_e5.csv"))).set_index("post_id")["nn"]
    demo_text = dict(zip(train_df["post_id"], all_posts[:N_TRAIN]))
    demo_label = dict(zip(train_df["post_id"], train_df["label"]))
    head_texts = []
    for pid in list(train_df["post_id"]) + list(test_df["post_id"]):
        demos, seen = [], set()
        for d in knn[pid].split():                      # nearest first; skip repeated texts
            if d in demo_text and demo_text[d] not in seen and len(demos) < args.fewshot:
                demos.append(d)
                seen.add(demo_text[d])
        demos = demos[::-1]                             # nearest example last, right before the question
        turns = []
        for d in demos:
            turns.append({"role": "user", "content": USER_PROMPT.format(post=demo_text[d][:DEMO_CHARS])})
            turns.append({"role": "assistant", "content": f"{ANSWER_PREFIX} {demo_label[d]}"})
        turns.append({"role": "user", "content": USER_PROMPT.format(post=PLACEHOLDER)})
        text = render_chat(turns)
        assert text.count(PLACEHOLDER) == 1
        head_texts.append(text.split(PLACEHOLDER)[0])
    heads = tokenizer(head_texts, add_special_tokens=False)["input_ids"]
IDS = [h + p[:args.max_post_tokens] + tail_ids for h, p in zip(heads, post_ids)]   # train rows first, then test rows
LENGTHS = np.array([len(x) for x in IDS])

# class c  <->  first token of " <label word>" (what follows "Emotion:"); the six tokens must be different
label_tokens = [tokenizer(" " + c, add_special_tokens=False)["input_ids"] for c in CLASSES]
LABEL_IDS = [t[0] for t in label_tokens]
assert len(set(LABEL_IDS)) == len(CLASSES), f"label words share a first token: {LABEL_IDS}"
if any(len(t) > 1 for t in label_tokens):
    log("note: some label words are more than one token; only the first token is scored")
log(f"prompt: head {int(np.median([len(h) for h in heads]))} (few-shot {args.fewshot}) + post (median {int(np.median([len(p) for p in post_ids]))}, "
    f"max {max(len(p) for p in post_ids)}, truncated {n_trunc}) + tail {len(tail_ids)} tokens | "
    f"total length median {int(np.median(LENGTHS))}, max {LENGTHS.max()}")
log("label tokens: " + ", ".join(f"{c}->{tokenizer.convert_ids_to_tokens(i)!r}" for c, i in zip(CLASSES, LABEL_IDS)))
log("prompt as the model sees it:\n" + tokenizer.decode(IDS[0])[-1500:])


# ---- Batching ----
def train_batches(rows, batch_size, rng):
    """Shuffle, then group rows of similar length (less padding). The longest batch goes first so that an
    out-of-memory error shows up at step 1 instead of in the middle of an epoch."""
    rows = rng.permutation(np.asarray(rows))
    chunk = batch_size * 50
    rows = np.concatenate([c[np.argsort(LENGTHS[c], kind="stable")] for c in
                           (rows[i:i + chunk] for i in range(0, len(rows), chunk))])
    out = [rows[i:i + batch_size] for i in range(0, len(rows), batch_size)]
    out = [out[i] for i in rng.permutation(len(out))]
    longest = int(np.argmax([LENGTHS[b].max() for b in out]))
    out.insert(0, out.pop(longest))
    return out


def collate(rows):
    """Right-pad a batch of token id lists; returns input ids and attention mask on DEVICE."""
    width = int(LENGTHS[rows].max())
    input_ids = torch.full((len(rows), width), PAD_ID, dtype=torch.long)
    attention = torch.zeros((len(rows), width), dtype=torch.long)
    for r, i in enumerate(rows):
        input_ids[r, :LENGTHS[i]] = torch.tensor(IDS[i])
        attention[r, :LENGTHS[i]] = 1
    return input_ids.to(DEVICE), attention.to(DEVICE)


# ---- Model ----
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)


def build_model():
    """Frozen base model (4-bit on GPU) with fresh LoRA adapters. Returns (peft wrapper, underlying causal LM)."""
    kwargs = {"dtype": torch.float16 if DEVICE.type == "cuda" else torch.float32}
    if USE_4BIT:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16)
        kwargs["device_map"] = {"": 0}
    if args.device_map_auto and DEVICE.type == "cuda":
        kwargs["device_map"] = "auto"
        kwargs["max_memory"] = {i: int(args.max_gpu_gb * 2**30) for i in range(torch.cuda.device_count())}
    try:
        net = AutoModelForCausalLM.from_pretrained(args.model, **kwargs)
    except ValueError as err:              # e.g. Mistral3 (vision-language) has no causal-LM auto class
        log(f"AutoModelForCausalLM failed ({str(err)[:120]}) -> AutoModelForImageTextToText")
        from transformers import AutoModelForImageTextToText
        net = AutoModelForImageTextToText.from_pretrained(args.model, **kwargs)
    log(f"loaded class {type(net).__name__}")
    if "device_map" not in kwargs:
        net.to(DEVICE)
    for p in net.parameters():
        p.requires_grad_(False)
    net.config.use_cache = False
    if not args.no_grad_ckpt:
        net.gradient_checkpointing_enable()   # recompute activations in the backward pass: much less memory
        net.enable_input_require_grads()      # needed so gradients flow through checkpointed blocks of a frozen model
    if args.screen and args.no_probe:         # zero-shot only: no adapters needed
        return net, net
    peft_model = get_peft_model(net, LoraConfig(
        r=args.lora_r, lora_alpha=args.lora_alpha, lora_dropout=args.lora_dropout, bias="none",
        target_modules=LORA_TARGETS, task_type="CAUSAL_LM"))
    return peft_model, peft_model.get_base_model()


def class_logits(net, input_ids, attention):
    """Scores of the six label tokens at the position where the answer would start (last real token)."""
    backbone = getattr(net, net.base_model_prefix)
    backbone = getattr(backbone, "language_model", backbone)   # vision-language wrappers: text model inside
    hidden = backbone(input_ids=input_ids, attention_mask=attention, use_cache=False).last_hidden_state
    last = (attention.sum(1) - 1).to(hidden.device)
    hidden_last = hidden[torch.arange(hidden.size(0), device=hidden.device), last]
    vocab_scores = net.get_output_embeddings()(hidden_last)
    return vocab_scores[:, LABEL_IDS].float()


@torch.no_grad()
def predict_proba(peft_model, net, rows):
    """Class probabilities for the given global row ids (train rows 0..N_TRAIN-1, test rows after that)."""
    peft_model.eval()
    rows = np.asarray(rows)
    order = rows[np.argsort(LENGTHS[rows], kind="stable")[::-1]]   # longest first
    probs = {}
    for i in range(0, len(order), args.eval_batch):
        batch = order[i:i + args.eval_batch]
        input_ids, attention = collate(batch)
        with autocast():
            logits = class_logits(net, input_ids, attention)
        p = torch.softmax(logits, dim=-1).cpu().numpy()
        for r, row in enumerate(batch):
            probs[int(row)] = p[r]
    out = np.stack([probs[int(r)] for r in rows])
    assert np.isfinite(out).all(), "non-finite probabilities (fp16 overflow?)"
    return out


def macro_f1(y_true, probs):
    return f1_score(y_true, probs.argmax(1), average="macro", labels=list(range(len(CLASSES))), zero_division=0)


def to_prob_frame(post_ids, probs):
    out = pd.DataFrame(probs, columns=CLASSES)
    out.insert(0, "post_id", np.asarray(post_ids))
    return out


# ---- One fold ----
def run_fold(fold):
    t0 = time.time()
    fold_seed = args.seed + fold
    seed_everything(fold_seed)
    rng = np.random.default_rng(fold_seed)
    tr_rows = np.where(fold_ids != fold)[0]
    va_rows = np.where(fold_ids == fold)[0]
    test_rows = np.arange(N_TRAIN, N_TRAIN + N_TEST)

    peft_model, net = build_model()
    params = [p for p in peft_model.parameters() if p.requires_grad]
    log(f"fold {fold}: model loaded in {time.time() - t0:.0f}s | trainable parameters {sum(p.numel() for p in params) / 1e6:.1f}M"
        + (f" | GPU memory {torch.cuda.memory_allocated() / 2**30:.1f} GB" if DEVICE.type == "cuda" else ""))

    zero_shot = macro_f1(y[va_rows], predict_proba(peft_model, net, va_rows))   # adapters start as a no-op
    log(f"fold {fold}: zero-shot val macro F1 {zero_shot:.4f} (before any training)")

    optimizer = torch.optim.AdamW(params, lr=args.lr, weight_decay=args.weight_decay)
    steps_per_epoch = int(np.ceil(np.ceil(len(tr_rows) / args.batch) / args.accum))
    total_steps = steps_per_epoch * args.epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(args.warmup * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    best_f1, best_epoch, best_val, best_test, history, bad_steps = -1.0, 0, None, None, [], 0
    for epoch in range(1, args.epochs + 1):
        t_ep = time.time()
        peft_model.train()
        batches = train_batches(tr_rows, args.batch, rng)
        losses = []
        optimizer.zero_grad(set_to_none=True)
        for step, rows in enumerate(batches):
            input_ids, attention = collate(rows)
            with autocast():
                logits = class_logits(net, input_ids, attention)
            loss = torch.nn.functional.cross_entropy(logits, torch.tensor(y[rows], device=logits.device))
            if not torch.isfinite(loss):
                bad_steps += 1
                optimizer.zero_grad(set_to_none=True)
                if bad_steps > MAX_BAD_STEPS:
                    raise RuntimeError("too many NaN/inf losses (fp16 overflow?)")
                continue
            scaler.scale(loss / args.accum).backward()
            losses.append(loss.item())
            if (step + 1) % args.accum == 0 or (step + 1) == len(batches):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            if step == 0 or (step + 1) % 20 == 0:
                done = sum(len(b) for b in batches[:step + 1])
                log(f"  fold {fold} epoch {epoch} step {step + 1}/{len(batches)}: loss {np.mean(losses[-20:]):.4f} | "
                    f"{done / (time.time() - t_ep):.1f} rows/s"
                    + (f" | peak GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GB" if DEVICE.type == "cuda" else ""))

        val_probs = predict_proba(peft_model, net, va_rows)
        val_f1 = macro_f1(y[va_rows], val_probs)
        val_acc = accuracy_score(y[va_rows], val_probs.argmax(1))
        improved = val_f1 > best_f1
        history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), "val_macro_f1": float(val_f1),
                        "val_acc": float(val_acc), "minutes": (time.time() - t_ep) / 60})
        log(f"fold {fold} epoch {epoch}: train loss {np.mean(losses):.4f} | val macro F1 {val_f1:.4f} | "
            f"val acc {val_acc:.4f} | {(time.time() - t_ep) / 60:.1f} min" + (" *" if improved else ""))
        if improved:
            best_f1, best_epoch, best_val = val_f1, epoch, val_probs
            best_test = predict_proba(peft_model, net, test_rows)

    name = f"{args.tag}_s{args.seed}_f{fold}"
    to_prob_frame(train_df["post_id"].to_numpy()[va_rows], best_val).to_csv(OUT_DIR / f"val_{name}.csv", index=False)
    to_prob_frame(test_df["post_id"], best_test).to_csv(OUT_DIR / f"testfold_{name}.csv", index=False)
    info = {"tag": args.tag, "seed": args.seed, "fold": fold, "model": args.model, "best_epoch": best_epoch,
            "best_val_macro_f1": float(best_f1), "zero_shot_val_macro_f1": float(zero_shot), "history": history,
            "bad_steps": bad_steps, "minutes": (time.time() - t0) / 60, "n_train": int(len(tr_rows)),
            "n_val": int(len(va_rows)), "settings": {k: v for k, v in vars(args).items()}}
    json.dump(info, open(OUT_DIR / f"log_{name}.json", "w", encoding="utf-8"), indent=2)
    log(f"fold {fold} DONE: best epoch {best_epoch}, val macro F1 {best_f1:.4f}, {(time.time() - t0) / 60:.1f} min "
        f"-> {OUT_DIR / f'val_{name}.csv'}")

    del peft_model, net, optimizer, scheduler, scaler, params
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def run_screen():
    """Model screening: zero-shot quality on all rows, then a few training steps per batch size (memory, speed)."""
    t0 = time.time()
    seed_everything(args.seed)
    peft_model, net = build_model()
    loaded_gb = sum(torch.cuda.memory_allocated(i) for i in range(torch.cuda.device_count())) / 2**30 if DEVICE.type == "cuda" else 0.0
    log(f"screen {args.tag}: model loaded in {time.time() - t0:.0f}s | GPU memory {loaded_gb:.1f} GB (all GPUs)")

    t1 = time.time()
    probs = predict_proba(peft_model, net, np.arange(N_TRAIN + N_TEST))   # adapters are still a no-op: zero-shot
    zs_f1 = macro_f1(y, probs[:N_TRAIN])
    zs_acc = accuracy_score(y, probs[:N_TRAIN].argmax(1))
    per_fold = [float(macro_f1(y[fold_ids == k], probs[:N_TRAIN][fold_ids == k])) for k in sorted(np.unique(fold_ids))]
    log(f"screen {args.tag}: zero-shot macro F1 {zs_f1:.4f} | acc {zs_acc:.4f} | per fold "
        + " ".join(f"{v:.4f}" for v in per_fold) + f" | {(N_TRAIN + N_TEST) / (time.time() - t1):.1f} rows/s")
    if zs_f1 < 0.25:
        log(f"WARNING {args.tag}: zero-shot macro F1 {zs_f1:.4f} is near chance; weights may not have loaded correctly")
    to_prob_frame(train_df["post_id"], probs[:N_TRAIN]).to_csv(OUT_DIR / f"oof_zs_{args.tag}.csv", index=False)
    to_prob_frame(test_df["post_id"], probs[N_TRAIN:]).to_csv(OUT_DIR / f"test_zs_{args.tag}.csv", index=False)

    probe = {}
    if not args.no_probe:
        params = [p for p in peft_model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(params, lr=args.lr, weight_decay=args.weight_decay)
        scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
        rng = np.random.default_rng(args.seed)
    for batch_size in ([] if args.no_probe else [int(b) for b in args.probe_batches.split(",")]):
        try:
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
            peft_model.train()
            timings, losses = [], []
            for rows in train_batches(np.arange(N_TRAIN), batch_size, rng)[:5]:   # the first one is the longest batch
                t_step = time.time()
                input_ids, attention = collate(rows)
                with autocast():
                    logits = class_logits(net, input_ids, attention)
                loss = torch.nn.functional.cross_entropy(logits, torch.tensor(y[rows], device=logits.device))
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                if DEVICE.type == "cuda":
                    torch.cuda.synchronize()
                timings.append((len(rows), time.time() - t_step))
                losses.append(float(loss.item()))
            rows_per_s = sum(n for n, _ in timings[1:]) / sum(t for _, t in timings[1:])
            peak = torch.cuda.max_memory_allocated() / 2**30 if DEVICE.type == "cuda" else 0.0
            probe[batch_size] = {"peak_gb": peak, "rows_per_s": rows_per_s, "finite_loss": bool(np.isfinite(losses).all())}
            log(f"screen {args.tag}: batch {batch_size}: peak GPU {peak:.1f} GB | {rows_per_s:.1f} rows/s | "
                f"losses {' '.join(f'{v:.2f}' for v in losses)}")
        except RuntimeError as err:
            if "out of memory" not in str(err).lower():
                raise
            probe[batch_size] = {"oom": True}
            log(f"screen {args.tag}: batch {batch_size}: OUT OF MEMORY")
            optimizer.zero_grad(set_to_none=True)
            gc.collect()
    json.dump({"tag": args.tag, "model": args.model, "zero_shot_macro_f1": float(zs_f1), "zero_shot_acc": float(zs_acc),
               "zero_shot_per_fold": per_fold, "loaded_gb": loaded_gb, "probe": probe, "minutes": (time.time() - t0) / 60},
              open(OUT_DIR / f"screen_{args.tag}.json", "w", encoding="utf-8"), indent=2)


if args.screen:
    run_screen()
else:
    for fold in FOLDS:
        run_fold(fold)
log("worker finished")

In [ ]:
# [AI draft]
# 3.6a Load the 17 untrained-LLM runs and build the committee (temperature-softened average)
def soften(P, t):
    Q = np.exp(np.log(np.clip(P, 1e-9, 1)) / t)
    return Q / Q.sum(1, keepdims=True)

LLM_OOF, LLM_TEST = {}, {}
for p in sorted(glob.glob(str(RESEARCH / "llm" / "out" / "*" / "preds" / "oof_zs_*.csv"))):
    tag = Path(p).stem[len("oof_zs_"):]
    if tag not in LLM_OOF and Path(p.replace("oof_zs_", "test_zs_")).exists():
        LLM_OOF[tag], LLM_TEST[tag] = read_probs(p, train_ids), read_probs(p.replace("oof_zs_", "test_zs_"), test_ids)
assert len(LLM_OOF) == 17, sorted(LLM_OOF)
TEMPS = [1, 2, 3, 5]
COMM = {t: np.mean([soften(LLM_OOF[n], t) for n in sorted(LLM_OOF)], 0) for t in TEMPS}
COMM_TEST = {t: np.mean([soften(LLM_TEST[n], t) for n in sorted(LLM_TEST)], 0) for t in TEMPS}
display(pd.DataFrame({"run": sorted(LLM_OOF), "OOF macro F1": [round(macro_f1(y_int, LLM_OOF[n].argmax(1)), 4) for n in sorted(LLM_OOF)]})
        .sort_values("OOF macro F1", ascending=False).reset_index(drop=True))
print(f"committee (T=2) alone: OOF macro F1 {macro_f1(y_int, COMM[2].argmax(1)):.4f}")

### 3.7 Pseudo-labels: teacher for the twrob_pl student
For fold k the teacher is 0.7 x (QLoRA Qwen2.5-7B trained on the other four folds, test probabilities) + 0.3 x
(untrained Qwen2.5-14B, softened): neither part saw the labels of fold k, so the student's fold-k OOF score stays
honest. The student trains on the train folds plus the 1,440 **test texts** with these soft labels (test labels are
never used). The cell only writes the script that built `teacher_f<k>.csv` (it ran from `research/llm/`).

In [ ]:
%%writefile data/kaggle/jobs/make_teacher.py
# [AI draft]
"""Teacher files for pseudo-labelling (round 2): teacher_f<k>.csv = test probabilities from models that never saw
the labels of fold k.

  teacher_k = (1 - a) * [fine-tuned LLM of fold k, test probabilities]  +  a * [zero-shot LLM, softened]
The fine-tuned fold-k model was trained on the other four folds only; the zero-shot model saw no labels at all,
so both parts are leak-free for a student that is validated on fold k.
Usage: python make_teacher.py [FT_TAG] [SEED] [ZS_TAG] [A]      default: qwen25_7b 42 qwen25_14b 0.3
Writes research/llm/teacher/ (+ dataset-metadata.json for a private Kaggle dataset). Nothing is uploaded.
"""
import glob, json, sys
from pathlib import Path
import numpy as np, pandas as pd

sys.stdout.reconfigure(encoding="utf-8")
FT = sys.argv[1] if len(sys.argv) > 1 else "qwen25_7b"; SEED = int(sys.argv[2]) if len(sys.argv) > 2 else 42
ZS = sys.argv[3] if len(sys.argv) > 3 else "qwen25_14b"; A = float(sys.argv[4]) if len(sys.argv) > 4 else 0.3
ROOT = Path(__file__).resolve().parents[1]; OUT = ROOT / "llm" / "out"
C = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]


def soften(P, t=3):
    Q = np.exp(np.log(np.clip(P, 1e-9, 1)) / t); return Q / Q.sum(1, keepdims=True)


zs = pd.read_csv(OUT / "dm2026-lab2-llm-screen" / "preds" / f"test_zs_{ZS}.csv").set_index("post_id")
d = ROOT / "llm" / "teacher"; d.mkdir(exist_ok=True)
for k in range(5):
    hits = glob.glob(str(OUT / "*" / "preds" / f"testfold_{FT}_s{SEED}_f{k}.csv"))
    assert len(hits) == 1, f"fold {k}: expected one fine-tuned test file, found {len(hits)}"
    ft = pd.read_csv(hits[0]).set_index("post_id")
    P = (1 - A) * ft[C].to_numpy() + A * soften(zs.loc[ft.index, C].to_numpy())
    t = pd.DataFrame(P / P.sum(1, keepdims=True), columns=C); t.insert(0, "post_id", ft.index.to_numpy())
    t.to_csv(d / f"teacher_f{k}.csv", index=False)
    print(f"fold {k}: {len(t)} rows | mean max prob {P.max(1).mean():.3f} | rows with max prob >= 0.8: {(P.max(1) >= 0.8).mean():.2f} "
          f"| agreement fine-tuned vs zero-shot {(ft[C].to_numpy().argmax(1) == zs.loc[ft.index, C].to_numpy().argmax(1)).mean():.3f}")
json.dump({"title": "dm2026-lab2-pl-teacher", "id": "trex9527/dm2026-lab2-pl-teacher", "licenses": [{"name": "other"}]},
          open(d / "dataset-metadata.json", "w"), indent=2)
print("wrote", d)

### 3.8 A member from a different lineage: PPMd class-corpus compression (CPU)
Not every useful model has to grow towards transformers. Compression-based classifiers (PPM, Bratko et al. 2006)
read text one character at a time and predict the next byte from the previous ones; a post that the model trained on
one class's posts can compress cheaply "looks like" that class.
* Each class's training posts are joined into a corpus and cut into **equal 16 KB chunks** (remainder dropped) so that
  every class gets the same amount of context - otherwise big classes (joy) win everywhere.
* Cost of a post for class c = extra bytes PPMd (order 6) needs after each chunk of c, averaged over the chunks.
* Probability ~ class prior x exp(-beta x cost); beta is picked on the training folds (it came out 0.5).
* Out-of-fold: the corpora of fold k use the other four folds only; for test, train texts that also occur in test are
  removed from the corpora.
Alone it reaches macro F1 0.40 - weak, but it agrees with the final blend on only 47% of the posts.
`pyppmd` is installed outside the virtual environment (`research/llm/_pylib`). The costs take ~4 minutes; by default
the cached costs are loaded (set `RECOMPUTE_PPMD = True` to recompute them).

In [ ]:
# [AI draft]
# 3.8a PPMd class-corpus costs (equal 16 KB chunks per class), out-of-fold for train, all-train corpora for test
import sys
sys.path.insert(0, str(RESEARCH / "llm" / "_pylib"))
RECOMPUTE_PPMD = False
PPMD_CACHE = RESEARCH / "error_analysis_final" / "ppmd_class_costs.npz"
CHUNK = 16_000
low = lambda s: s.astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
t_train, t_test = low(train_df["text"]).to_numpy(), low(test_df["text"]).to_numpy()

def ppmd_len(b):
    import pyppmd
    enc = pyppmd.Ppmd7Encoder(6, 16 << 20)
    return len(enc.encode(b) + enc.flush())

def class_chunks(rows):
    """Equal-size chunks (bytes) of each class corpus; the remainder shorter than a chunk is dropped."""
    out = []
    for c in range(len(CLASSES)):
        cur, lst = b"", []
        for t in t_train[rows[y_int[rows] == c]]:
            cur += (t + "\n").encode("utf-8")
            if len(cur) >= CHUNK:
                lst.append(cur); cur = b""
        out.append([(ch, ppmd_len(ch)) for ch in lst])
    return out

def ppmd_costs(chunks, queries):
    S = np.zeros((len(queries), len(CLASSES)))
    for i, q in enumerate(queries):
        b = (q + "\n").encode("utf-8")
        for c, lst in enumerate(chunks):
            S[i, c] = np.mean([ppmd_len(ch + b) - base for ch, base in lst])
    return S

if RECOMPUTE_PPMD or not PPMD_CACHE.exists():
    COST_OOF = np.zeros((len(train_df), len(CLASSES)))
    for k in range(5):
        COST_OOF[fold_all == k] = ppmd_costs(class_chunks(np.where(fold_all != k)[0]), t_train[fold_all == k])
    keep = np.where(~np.isin(t_train, t_test))[0]
    COST_TEST = ppmd_costs(class_chunks(keep), t_test)
else:
    z = np.load(PPMD_CACHE)
    COST_OOF, COST_TEST = z["S_oof"], z["S_te"]
print("cost tables:", COST_OOF.shape, COST_TEST.shape, "| standalone (argmin cost, no prior) macro F1:",
      round(macro_f1(y_int, COST_OOF.argmin(1)), 4))

### 3.9 Ensembles A and B, nested CV, submissions
* **R6** = 0.7 x mean(5 transformers) + 0.15 x TF-IDF LR + 0.15 x bge LR (fixed weights).
* **A** = (1 - w) x R6 + w x committee(T).
* **B** = (1 - v) x A + v x PPMd member.
Only T, w (A) and T, w, v, beta (B) are chosen, on macro F1 / log-loss of the training folds. **Nested CV**: choose on
4 folds, score the 5th, for every fold - the honest estimate. Nine learned ways of combining the members (stacking,
Super Learner, gating, ...) were tried and none beat this fixed formula (3.10). Final Kaggle picks: **B** (main) and
**A** (fallback).

In [ ]:
# [AI draft]
# 3.9a Members on the same rows, then A and B with nested CV
TF_OOF, TF_TEST = (read_probs(RESEARCH / "preds_backup" / "ens2" / f"{s}_tfidf_lr.csv", ids) for s, ids in [("oof", train_ids), ("test", test_ids)])
BGE_OOF, BGE_TEST = (read_probs(RESEARCH / "preds_backup" / "ens2" / f"{s}_bge_lr.csv", ids) for s, ids in [("oof", train_ids), ("test", test_ids)])
# the ens2 files were produced by cells 3.1c, 3.3 and 3.4 of this notebook; check against them if they were run here
for name, P, f in [("TF-IDF", TF_OOF, "oof_tfidf_lr.csv"), ("bge", BGE_OOF, "oof_bge_lr.csv")]:
    if (pred_dir / f).exists():
        print(f"{name}: OOF from this notebook identical to the one used:", np.allclose(read_probs(pred_dir / f, train_ids), P, atol=1e-6))
R6 = 0.7 * np.mean([OOF_TR[k] for k in TOP5], 0) + 0.15 * TF_OOF + 0.15 * BGE_OOF
R6_TEST = 0.7 * np.mean([TEST_TR[k] for k in TOP5], 0) + 0.15 * TF_TEST + 0.15 * BGE_TEST
WS, VS, BETAS = [0.1, 0.15, 0.2, 0.25, 0.3, 0.4], [0, 0.05, 0.1, 0.15], [0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 8 * np.log(2)]

def ppmd_probs(cost, prior, beta):
    L = -beta * cost + np.log(prior)[None, :]
    P = np.exp(L - L.max(1, keepdims=True))
    return P / P.sum(1, keepdims=True)

def fit_ppmd(rows):
    """Class prior and beta (log-loss) from the given training rows."""
    prior = np.bincount(y_int[rows], minlength=len(CLASSES)) / len(rows)
    nll = lambda b: -np.mean(np.log(np.clip(ppmd_probs(COST_OOF[rows], prior, b)[np.arange(len(rows)), y_int[rows]], 1e-12, None)))
    return prior, min(BETAS, key=nll)

blend = lambda R, Cm, M, w, v: (1 - v) * ((1 - w) * R + w * Cm) + v * M
def choose(rows, M, vs):
    return max(((t, w, v) for t in TEMPS for w in WS for v in vs),
               key=lambda c: macro_f1(y_int[rows], blend(R6[rows], COMM[c[0]][rows], M[rows], c[1], c[2]).argmax(1)))

M_OOF = np.zeros_like(R6)                       # PPMd probabilities: fold k uses prior/beta from the other folds
pred_A, pred_B, rows_out = np.zeros(len(y_int), int), np.zeros(len(y_int), int), []
for k in range(5):
    tr, va = np.where(fold_all != k)[0], np.where(fold_all == k)[0]
    prior, beta = fit_ppmd(tr)
    M = np.zeros_like(R6); M[tr] = ppmd_probs(COST_OOF[tr], prior, beta); M[va] = ppmd_probs(COST_OOF[va], prior, beta)
    M_OOF[va] = M[va]
    tA, wA, _ = choose(tr, M, [0]); pred_A[va] = blend(R6[va], COMM[tA][va], M[va], wA, 0).argmax(1)
    tB, wB, vB = choose(tr, M, VS); pred_B[va] = blend(R6[va], COMM[tB][va], M[va], wB, vB).argmax(1)
    rows_out.append({"fold": k, "A: T, w": (tA, wA), "A macro F1": round(macro_f1(y_int[va], pred_A[va]), 4),
                     "B: T, w, v, beta": (tB, wB, vB, round(beta, 2)), "B macro F1": round(macro_f1(y_int[va], pred_B[va]), 4)})
display(pd.DataFrame(rows_out))
rng = np.random.default_rng(0); boots = [rng.integers(0, len(y_int), len(y_int)) for _ in range(2000)]
gain = np.array([macro_f1(y_int[b], pred_B[b]) - macro_f1(y_int[b], pred_A[b]) for b in boots])
print(f"NESTED CV macro F1  A {macro_f1(y_int, pred_A):.4f} (acc {np.mean(pred_A == y_int):.4f}) | "
      f"B {macro_f1(y_int, pred_B):.4f} (acc {np.mean(pred_B == y_int):.4f})")
print(f"B - A: {macro_f1(y_int, pred_B) - macro_f1(y_int, pred_A):+.4f}, bootstrap 95% [{np.percentile(gain, 2.5):+.4f}, "
      f"{np.percentile(gain, 97.5):+.4f}], P(B better) {(gain > 0).mean():.2f}")

In [ ]:
# [AI draft]
# 3.9b Final choices on all train rows, test predictions, submissions (header id,label) + checks
def write_submission(labels, name):
    sub = sample_sub[["post_id"]].merge(pd.DataFrame({"post_id": test_ids, "label": np.asarray(CLASSES)[labels]}),
                                        on="post_id", how="left", validate="one_to_one")
    assert sub.shape == (1440, 2) and sub["label"].notna().all() and sub["label"].isin(CLASSES).all()
    sub = sub.rename(columns={"post_id": "id"})
    sub.to_csv(pred_dir / name, index=False)
    return sub

allr = np.arange(len(y_int))
tA, wA, _ = choose(allr, M_OOF, [0])
sub_A = write_submission(blend(R6_TEST, COMM_TEST[tA], 0, wA, 0).argmax(1), "submission_A.csv")
prior_all, beta_all = fit_ppmd(allr)
tB, wB, vB = choose(allr, M_OOF, VS)
M_TEST = ppmd_probs(COST_TEST, prior_all, beta_all)
sub_B = write_submission(blend(R6_TEST, COMM_TEST[tB], M_TEST, wB, vB).argmax(1), "submission_B.csv")
print(f"A: T={tA}, w={wA} | B: T={tB}, w={wB}, v={vB}, beta={beta_all:.2f} | rows where A and B differ: {(sub_A['label'] != sub_B['label']).sum()}")
for sub, f in [(sub_A, "submission_R6pl_ls5_llm_all.csv"), (sub_B, "submission_R6pl_ls5_ppmd_prior.csv")]:
    sent = pd.read_csv(RESEARCH / "final" / f)
    print(f"identical to the file submitted to Kaggle ({f}):", sent.equals(sub))
display(pd.DataFrame({"train %": (train_df["label"].value_counts(normalize=True) * 100).round(1),
                      "B test pred %": (sub_B["label"].value_counts(normalize=True) * 100).round(1)}))

### 3.10 Tried and rejected (nested CV, gain vs. A = 0.6447)
Adoption rule fixed before each run: nested mean gain >= +0.005 and positive in at least 4 of 5 folds. Changing only
one member's seed moves the blend between 0.641 and 0.648, so smaller gains are noise. Scripts are in `research/llm/`.

| idea | gain | why it did not help |
|---|---|---|
| Caruana hill climbing over 30 members | -0.004 | optimises F1 directly; 0.6503 in-sample but overfits |
| logistic-regression stacker (378 parameters) | -0.015 | too many parameters for 2,700 noisy rows |
| gradient-boosted tree stacker | -0.001 | tie |
| Super Learner (log-loss / soft F1 / class-balanced log-loss) | -0.002 / -0.014 / -0.003 | log-loss drops useful members; soft F1 rewards one over-confident member |
| global group weights; per-post gating of the 3 groups | +0.0005; +0.0002 | tie - no stable "when to trust whom" signal |
| swapping the classical members (any 1-2 of 36 saved ones) | -0.0026 | each fold picks a different pair: chasing noise |
| same-label concatenation augmentation | -0.0032 per seed | the model memorises more easily |
| copying the train label of identical test texts | -0.0028 (proxy) | copied labels are right 68% of the time, the model 74% |
| compression distance + kNN; zlib class corpora; character n-gram LM | +0.0009; -0.0008; -0.0031 | weaker than PPMd |
| QLoRA fine-tuned Qwen2.5-7B as a member | worse | too similar to the transformers (kept as teacher only) |
| PPMd member, B (kept as the main pick) | **+0.0042** (4/5 folds, P = 0.96) | below the +0.005 rule; chosen as a deliberate bet with A as fallback |